# Bài thực hành làm sạch và phân tích dữ liệu đặt vé máy bay — Notebook khung

**Phiên bản Airline 1.0 · Dành cho sinh viên · Chưa có lời giải**

| Thông tin nhóm | Điền vào đây |
|---|---|
| Mã nhóm | NhomXX |
| Thành viên 1 (MSSV) — phần phụ trách | … |
| Thành viên 2 (MSSV) — phần phụ trách | … |
| Thành viên 3 (MSSV) — phần phụ trách | … |
| Ngày chạy lại toàn bộ lần cuối | … |

**Trước khi nộp:** đổi tên tệp thành `airline_analysis.ipynb`.

## Cách dùng notebook này

Notebook đi theo đúng 5 giai đoạn của đề bài: **A Khảo sát → B Làm sạch → C Bảng sạch & kiểm tra → D Phân tích 10 câu → E Xuất & nộp**.

| Ký hiệu | Ý nghĩa | Bạn cần làm gì |
|---|---|---|
| 📋 Thẻ nhiệm vụ | Bảng đầu mỗi mục: mục tiêu, đầu vào/ra, các bước, bẫy | Đọc trước khi viết mã |
| `# TODO` | Chỗ nhóm phải viết mã | Thay bằng mã của nhóm; xóa chữ TODO khi xong |
| `= None` | Biến đầu ra nhóm phải tạo | Gán DataFrame thật, **giữ đúng tên biến** |
| ✅ / ❌ | Kết quả ô tự kiểm tra | ❌ nghĩa là chưa xong hoặc chưa đúng quy ước — đọc gợi ý in kèm |
| ✍️ | Ô văn bản nhóm phải viết | Viết bằng số liệu thật, có mẫu số |

**Ba quy tắc vàng**

1. **Không sửa tay** `airline_bookings_dirty.csv`, không thay bằng bộ dữ liệu gốc sạch.
2. **Không xóa các ô kiểm tra** (✅/❌). Chúng giúp bạn và giảng viên thấy tiến độ.
3. Cuối mỗi buổi: **Runtime/Kernel → Restart & Run all**. Notebook phải chạy được từ đầu đến cuối.

Notebook vẫn chạy hết khi còn dang dở: các ô kiểm tra chỉ in ❌, không làm dừng chương trình.

## 0 · Thiết lập

**Colab:** File → Upload notebook; mở bảng 📁 Files, tải `airline_bookings_dirty.csv` lên. Có thể tải thêm `kiem_tra_nop_bai.py` để tự kiểm tra ở cuối.
**Máy cá nhân:** giữ cấu trúc thư mục của gói bài tập (notebook ở thư mục gốc, dữ liệu trong `data/`).

Ô 0.1 chứa công cụ hỗ trợ dùng chung. **Không cần sửa**, chỉ cần chạy.

In [4]:
# 0.1 CÔNG CỤ HỖ TRỢ — KHÔNG CẦN SỬA
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 50)

# ---- Đường dẫn: tự tìm dữ liệu ở data/ hoặc cùng thư mục notebook (Colab) ----
CANDIDATES = [Path('data/airline_bookings_dirty.csv'), Path('airline_bookings_dirty.csv')]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
OUTPUT_DIR = Path('outputs')
for sub in ['', 'figures', 'tables']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ---- Quy ước của đề (Mục 5) ----
SNAPSHOT_DATE = pd.Timestamp('2026-09-15')
MISSING_TOKENS = ['NULL', 'N/A', 'NA', '?', 'unknown', '-']
AIRLINE_COLORS = {'Aurora Air': '#2F4858', 'Horizon Wings': '#E07A1F'}
STATUS_COLORS = {'Flown': '#5B8E7D', 'Cancelled': '#B81D24', 'No-Show': '#F2A541'}
FIGSIZE = (9, 5)

AIRLINES = ['Aurora Air', 'Horizon Wings']
MEALS = ['Standard', 'Vegetarian', 'Special', 'No Meal']
SEGMENTS = ['Direct', 'Online OTA', 'Offline Agent', 'Corporate', 'Group', 'Package Tour', 'Loyalty']
CHANNELS = ['Website', 'Mobile App', 'Travel Agency', 'GDS/Corporate']
FARE_POLICIES = ['Flexible', 'Semi-Flexible', 'Non-Refundable']
TRAVELERS = ['Individual', 'Linked Booking', 'Corporate Contract', 'Group']
STATUSES = ['Flown', 'Cancelled', 'No-Show']
CABIN_CODES = ['E1', 'E2', 'E3', 'E4', 'P1', 'P2', 'B1', 'B2', 'B3', 'F1', 'F2', 'F3']
LEAD_GROUPS = ['0-7', '8-30', '31-90', '91-180', '181+']

DEPARTURE_MIN, DEPARTURE_MAX = pd.Timestamp('2024-07-01'), pd.Timestamp('2026-08-31')
STATUS_MIN, STATUS_MAX = pd.Timestamp('2024-01-01'), pd.Timestamp('2026-09-15')
LEAD_MIN, LEAD_MAX = 0, 365
WEEKEND_MAX, WEEK_MAX = 20, 50
PASSENGER_MAX = 10
FARE_MIN, FARE_MAX = 0, 5000
REQ_MAX = 5
N_RAW_ROWS = 10325

CLEAN_REQUIRED = ['reservation_id', 'airline', 'is_cancelled', 'booking_lead_days', 'departure_date', 'trip_weekend_days',
                  'trip_weekdays', 'adult_passengers', 'child_passengers', 'infant_passengers', 'meal_option',
                  'passenger_country', 'market_segment', 'sales_channel', 'is_returning_customer', 'booked_cabin_code',
                  'assigned_cabin_code', 'fare_policy', 'traveler_type', 'avg_ticket_fare', 'special_service_requests',
                  'booking_status', 'booking_status_date', 'departure_year', 'departure_month', 'total_trip_days',
                  'total_passengers', 'lead_time_group', 'cabin_changed_flag', 'booking_value_est']
SUMMARY_COLUMNS = ['airline', 'departure_year', 'departure_month', 'n_reservations', 'n_cancelled',
                   'cancel_rate_pct', 'passengers_flown']
LOG_COLUMNS = ['step', 'field', 'issue', 'detection_rule', 'action', 'affected_rows',
               'changed_cells', 'removed_rows', 'example_record_ids', 'reason', 'check_after']
LINEAGE_COLUMNS = ['record_id', 'reservation_id', 'status']
LINEAGE_STATUS = ['kept', 'merged', 'dropped']
QUALITY_COLUMNS = ['column', 'stage', 'denominator_rows', 'missing_count', 'invalid_count', 'missing_pct']

CORE_FILES = ['reservations_clean.csv', 'monthly_summary.csv', 'cleaning_log.csv', 'row_lineage.csv', 'quality_report.csv']
FIGURE_FILES = ['fig01_airline_cancel.png', 'fig02_monthly_departures_2025.png', 'fig03_leadtime_cancel.png',
                'fig04_top10_country.png', 'fig05_segment_cancel.png', 'fig06_fare_policy_status.png',
                'fig07_fare_hist.png', 'fig08_trip_days.png']
TABLE_FILES = ['a_missing_survey.csv', 'a_issue_catalog.csv', 'q01_before_after.csv',
               'q02_airline_cancel.csv', 'q03_monthly_departures_2025.csv', 'q04_leadtime_stats.csv',
               'q04_leadtime_cancel.csv', 'q05_top10_country.csv', 'q06_segment_cancel.csv',
               'q07_fare_policy_status.csv', 'q08_fare_stats_by_airline.csv', 'q09_trip_days.csv',
               'q10_cabin_change_by_airline.csv', 'q10_cabin_change_by_booked_code.csv']


def check(label, fn, hint=''):
    try:
        ok = bool(fn())
        detail = ''
    except Exception as e:
        ok = False
        detail = f'{type(e).__name__}: {e}'
    if ok:
        print('✅ ' + label)
    else:
        extra = (' — ' + detail) if detail else ((' — ' + hint) if hint else '')
        print('❌ ' + label + extra)
    return ok


def clean_key(x):
    if pd.isna(x): return pd.NA
    return re.sub(r'[^a-z]', '', str(x).strip().lower())


def log_step(step, field, issue, detection_rule, action, affected_rows=0, changed_cells=0,
             removed_rows=0, example_record_ids='', reason='', check_after=''):
    cleaning_log_rows.append({
        'step': step, 'field': field, 'issue': issue, 'detection_rule': detection_rule,
        'action': action, 'affected_rows': int(affected_rows), 'changed_cells': int(changed_cells),
        'removed_rows': int(removed_rows), 'example_record_ids': str(example_record_ids),
        'reason': reason, 'check_after': check_after,
    })


def save_table(frame, name):
    if frame is None:
        print(f'⏳ Chưa lưu {name}.csv: biến còn None.')
        return
    frame.to_csv(OUTPUT_DIR / 'tables' / f'{name}.csv', index=False, encoding='utf-8-sig')
    print(f'💾 outputs/tables/{name}.csv ({len(frame)} dòng)')

# ---- Nhập bộ kiểm tra cùng logic với tools/kiem_tra_nop_bai.py nếu file tồn tại ----
try:
    import importlib.util
    checker_candidates = [Path('tools/kiem_tra_nop_bai.py'), Path('kiem_tra_nop_bai.py')]
    checker_path = next((p for p in checker_candidates if p.exists()), None)
    if checker_path:
        spec = importlib.util.spec_from_file_location('airline_checker', checker_path)
        airline_checker = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(airline_checker)
        validate_outputs = airline_checker.validate_outputs
        print_report = airline_checker.print_report
    else:
        validate_outputs = print_report = None
except Exception as e:
    validate_outputs = print_report = None
    print('⚠️ Chưa nạp được checker:', e)

cleaning_log_rows = []
print('✅ Thiết lập xong. DATA_PATH =', DATA_PATH)


def save_fig(fig, name):
    """Lưu hình PNG 150 dpi vào outputs/figures/<name>.png."""
    if name + '.png' not in FIGURE_FILES:
        print(f'⚠️ Tên {name}.png không có trong danh sách chuẩn: kiểm tra lại chính tả.')
    fig.savefig(OUTPUT_DIR / 'figures' / f'{name}.png', dpi=150, bbox_inches='tight')
    print(f'🖼️ outputs/figures/{name}.png')


def finish_chart(ax, title, xlabel, ylabel, note=None):
    """Định dạng chuẩn: tiêu đề, nhãn trục, ghi chú mẫu số/nguồn dưới hình."""
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    if note:
        ax.figure.text(0.01, -0.04, note, fontsize=9, color='#555555', ha='left', va='top', wrap=True)


def show_progress():
    """Bảng tiến độ: các tệp bắt buộc đã có trong outputs/ hay chưa."""
    items = [('Dữ liệu', f, OUTPUT_DIR / f) for f in CORE_FILES] + \
            [('Bảng', f, OUTPUT_DIR / 'tables' / f) for f in TABLE_FILES] + \
            [('Hình', f, OUTPUT_DIR / 'figures' / f) for f in FIGURE_FILES]
    t = pd.DataFrame([(g, f, '✅' if q.exists() else '⬜') for g, f, q in items], columns=['Nhóm', 'Tệp', 'Có'])
    done = (t['Có'] == '✅').sum()
    print(f'Tiến độ tệp đầu ra: {done}/{len(t)} ({done / len(t):.0%})')
    return t


✅ Thiết lập xong. DATA_PATH = data\airline_bookings_dirty.csv


In [5]:
# 0.2 ĐỌC DỮ LIỆU — đọc mọi cột ở dạng chuỗi để thấy giá trị gốc
if not DATA_PATH.exists():
    raise FileNotFoundError('Không thấy airline_bookings_dirty.csv. Colab: tải tệp lên bảng Files. '
                            'Máy cá nhân: giữ tệp tại data/airline_bookings_dirty.csv.')

raw_df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
print('Kích thước đầu vào:', raw_df.shape)
assert raw_df.shape == (10325, 24), f'Đầu vào phải là 10.325 dòng × 24 cột, hiện tại: {raw_df.shape}'
assert raw_df['record_id'].is_unique, 'record_id phải duy nhất ở dữ liệu đầu vào.'
print('✅ Đầu vào đúng 10.325 dòng × 24 cột')
raw_df.head()


Kích thước đầu vào: (10325, 24)
✅ Đầu vào đúng 10.325 dòng × 24 cột


,record_id,reservation_id,airline,is_cancelled,booking_lead_days,departure_date,trip_weekend_days,trip_weekdays,adult_passengers,child_passengers,infant_passengers,meal_option,passenger_country,market_segment,sales_channel,is_returning_customer,booked_cabin_code,assigned_cabin_code,fare_policy,traveler_type,avg_ticket_fare,special_service_requests,booking_status,booking_status_date
0,R000001,AR002912,Aurora Air,0,19,2024-11-06,4,8,4,0,1,Special,VNM,Offline Agent,Travel Agency,0,F2,F2,Semi-Flexible,Individual,829.51,2,Flown,2024-11-07
1,R000002,AR008211,AURORA AIR,0,0,15-Nov-2024,2,4,1,0,0,Standard,USA,Online OTA,Website,1,E1,E1,Non-Refundable,Individual,151.68,0,Flown,2024-11-17
2,R000003,AR005564,Aurora Air,0,42,12/12/2024,0,2,2,0,0,Standard,VNM,Corporate,Travel Agency,0,E4,E4,Flexible,Group,146.28,1,Flown,2024-12-12
3,R000004,AR007528,Aurora Air,1,89,2024-08-20,0,4,1,0,0,Special,SWE,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,206.48,0,Cancelled,2024-06-07
4,R000005,AR007195,Aurora Air,0,18,10/02/2025,0,5,1,1,0,No Meal,GBR,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,186.89,1,Flown,2025-02-11


---
# GIAI ĐOẠN A · KHẢO SÁT TRƯỚC KHI SỬA
*Mục tiêu của cả giai đoạn: hiểu dữ liệu đang sai ở đâu, sai bao nhiêu — **chưa sửa gì cả**.*

## A · Khảo sát dữ liệu

| | |
|---|---|
| **Mục tiêu** | Lập bảng thiếu theo cột và danh mục **ít nhất 10 nhóm vấn đề**, mỗi nhóm có số dòng và tối đa 5 `record_id` ví dụ. |
| **Đầu vào → Đầu ra** | `raw_df` → `missing_survey`, `issue_catalog` |
| **Thời gian gợi ý** | ≈ 2 giờ |
| **Đọc kèm** | Đề bài Mục 6 |

**Các bước cần làm**

1. **A1** In `shape`, tên cột, `dtypes`, 5 dòng mẫu.
2. **A2** Với từng cột: đếm ô rỗng (sau `strip`) và ô là ký hiệu thiếu (`MISSING_TOKENS`, không phân biệt hoa thường). Lưu `missing_survey`: `column, n_rows, empty_count, token_count, missing_total, missing_pct`.
3. **A3** `value_counts()` cho: `airline, is_cancelled, meal_option, market_segment, sales_channel, fare_policy, traveler_type, booking_status, booked_cabin_code, assigned_cabin_code`.
4. **A4** Khảo sát mẫu định dạng của `departure_date, booking_status_date, booking_lead_days, trip_weekend_days, trip_weekdays, adult_passengers, child_passengers, infant_passengers, avg_ticket_fare`; xem các cách ghi `passenger_country`.
5. **A5** Đếm dòng trùng trên **23 cột nghiệp vụ** (bỏ `record_id`); đếm `reservation_id` lặp trước và sau `strip().upper()`.
6. **A6** Gom phát hiện vào `issue_catalog`: `issue_id, field, issue, detection_rule, affected_rows, example_record_ids`.

**Gợi ý hàm:** `value_counts(dropna=False)`, `str.strip`, `str.lower`, `isin`, `duplicated(subset=...)`, `str.match`, `str.contains`

**⚠️ Bẫy thường gặp**

- Gọi `isna()` ngay sau `keep_default_na=False` → ô trống vẫn là chuỗi `''`.
- Gọi `duplicated()` trên cả 24 cột → không thấy trùng tuyệt đối vì `record_id` luôn khác nhau.
- Gộp hai reservation khác `reservation_id` chỉ vì các cột còn lại giống nhau — đó có thể là hai giao dịch độc lập.


In [6]:
# A1 · Tổng quan
# TODO: in raw_df.shape, list(raw_df.columns), raw_df.dtypes và raw_df.head()
print(raw_df.shape)
print(raw_df.columns)
print(raw_df.dtypes)
display(raw_df.head())

(10325, 24)
Index(['record_id', 'reservation_id', 'airline', 'is_cancelled',
       'booking_lead_days', 'departure_date', 'trip_weekend_days',
       'trip_weekdays', 'adult_passengers', 'child_passengers',
       'infant_passengers', 'meal_option', 'passenger_country',
       'market_segment', 'sales_channel', 'is_returning_customer',
       'booked_cabin_code', 'assigned_cabin_code', 'fare_policy',
       'traveler_type', 'avg_ticket_fare', 'special_service_requests',
       'booking_status', 'booking_status_date'],
      dtype='object')
record_id                   object
reservation_id              object
airline                     object
is_cancelled                object
booking_lead_days           object
departure_date              object
trip_weekend_days           object
trip_weekdays               object
adult_passengers            object
child_passengers            object
infant_passengers           object
meal_option                 object
passenger_country           objec

,record_id,reservation_id,airline,is_cancelled,booking_lead_days,departure_date,trip_weekend_days,trip_weekdays,adult_passengers,child_passengers,infant_passengers,meal_option,passenger_country,market_segment,sales_channel,is_returning_customer,booked_cabin_code,assigned_cabin_code,fare_policy,traveler_type,avg_ticket_fare,special_service_requests,booking_status,booking_status_date
0,R000001,AR002912,Aurora Air,0,19,2024-11-06,4,8,4,0,1,Special,VNM,Offline Agent,Travel Agency,0,F2,F2,Semi-Flexible,Individual,829.51,2,Flown,2024-11-07
1,R000002,AR008211,AURORA AIR,0,0,15-Nov-2024,2,4,1,0,0,Standard,USA,Online OTA,Website,1,E1,E1,Non-Refundable,Individual,151.68,0,Flown,2024-11-17
2,R000003,AR005564,Aurora Air,0,42,12/12/2024,0,2,2,0,0,Standard,VNM,Corporate,Travel Agency,0,E4,E4,Flexible,Group,146.28,1,Flown,2024-12-12
3,R000004,AR007528,Aurora Air,1,89,2024-08-20,0,4,1,0,0,Special,SWE,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,206.48,0,Cancelled,2024-06-07
4,R000005,AR007195,Aurora Air,0,18,10/02/2025,0,5,1,1,0,No Meal,GBR,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,186.89,1,Flown,2025-02-11


In [7]:
# A2 · Bảng thiếu theo cột
missing_survey = None   # TODO: DataFrame 24 dòng, cột: column, n_rows, empty_count, token_count, missing_total, missing_pct
results = []
n_rows = len(raw_df)
for column in raw_df.columns:
    s = raw_df[column].astype("string").str.strip()
    empty_count = (s == "").sum()

    token_count = raw_df[column].str.strip().str.lower().isin([t.lower() for t in MISSING_TOKENS]).sum()

    missing_total = empty_count + token_count

    missing_pct = missing_total / n_rows * 100
    results.append({
        "column": column,
        "n_rows": n_rows,
        "empty_count": empty_count,
        "token_count": token_count,
        "missing_total": missing_total,
        "missing_pct": missing_pct
    })
missing_survey = pd.DataFrame(results)
display(missing_survey)


,column,n_rows,empty_count,token_count,missing_total,missing_pct
0,record_id,10325,0,0,0,0.000000
1,reservation_id,10325,0,0,0,0.000000
2,airline,10325,0,0,0,0.000000
3,is_cancelled,10325,0,0,0,0.000000
4,booking_lead_days,10325,0,0,0,0.000000
5,departure_date,10325,5,20,25,0.242131
6,trip_weekend_days,10325,1,11,12,0.116223
7,trip_weekdays,10325,1,10,11,0.106538
8,adult_passengers,10325,0,12,12,0.116223
9,child_passengers,10325,5,35,40,0.387409


In [8]:
# A3 · Nhãn của các cột phân loại
# TODO: value_counts cho từng cột phân loại
columns_to_count = [
    "airline",
    "is_cancelled",
    "meal_option",
    "market_segment",
    "sales_channel",
    "fare_policy",
    "traveler_type",
    "booking_status",
    "booked_cabin_code",
    "assigned_cabin_code"
]
for column in columns_to_count:
    print(raw_df[column].value_counts(dropna=False))

airline
Aurora Air         6637
Horizon Wings      3513
Aurora               35
 Aurora Air          24
aurora air           23
horizon wings        23
AURORA AIR           21
Horizon              19
HORIZON WINGS        18
 Horizon Wings       12
Name: count, dtype: int64
is_cancelled
0        7385
1        2817
no         28
False      27
0.0        21
 No        20
 Yes       10
1.0         8
yes         5
True        4
Name: count, dtype: int64
meal_option
Standard         6084
No Meal          1879
Vegetarian       1256
Special           944
Std                25
standard meal      19
 Standard          18
STANDARD           15
NO MEAL            13
Veg                 8
 No Meal            7
none                7
?                   6
 Vegetarian         5
VEG                 5
NULL                4
N/A                 4
 Special            4
vegetarian          3
-                   3
                    3
null                3
special meal        2
no_meal             2
SPECIAL

In [9]:
# A4 · Định dạng ngày, số, tiền và mã quốc gia
# TODO: mẫu hóa định dạng (thay chữ số bằng 9) và value_counts
cols_to_check = ['departure_date', 'booking_status_date', 'booking_lead_days', 
                 'trip_weekend_days', 'trip_weekdays', 'adult_passengers', 
                 'avg_ticket_fare']

for col in cols_to_check:
    # Thay chữ số bằng 9 để xem pattern
    print(raw_df[col].str.replace(r'\d', '9', regex=True).value_counts().head())

print(raw_df['passenger_country'].value_counts().head(10))

departure_date
9999-99-99       8365
99/99/9999        777
99-Aug-9999        59
July 99, 9999      55
99-Jul-9999        54
Name: count, dtype: int64
booking_status_date
9999-99-99    10298
                 26
not-a-date        1
Name: count, dtype: int64
booking_lead_days
99      6762
999     2516
9        935
 99       29
99.9      24
Name: count, dtype: int64
trip_weekend_days
9         10147
99          121
9 days       21
9.9          18
N/A           3
Name: count, dtype: int64
trip_weekdays
9         9553
99         719
9 days      22
9.9         13
NA           5
Name: count, dtype: int64
adult_passengers
9        10184
99          83
9.9         26
9 pax       19
N/A          3
Name: count, dtype: int64
avg_ticket_fare
999.99     8164
999.9       918
99.99       810
9999.99     105
99.9         81
Name: count, dtype: int64
passenger_country
VNM    2111
CHN     668
JPN     665
KOR     533
THA     524
USA     497
SGP     404
AUS     369
GBR     352
IND     351
Name: count, dtyp

In [10]:
# A5 · Bản ghi trùng và khóa
# TODO: đếm trùng trên 23 cột nghiệp vụ; đếm reservation_id lặp trước/sau chuẩn hóa; lưu vài ví dụ record_id
columns_23=[]
# trước chuẩn hóa
duplicate=raw_df.duplicated(subset='reservation_id').sum()
print(duplicate)

# sau chuẩn hóa
reservation_after=raw_df['reservation_id'].astype('string').str.strip().str.lower()
duplicate_after=reservation_after.duplicated().sum()
print(duplicate_after)

288
325


In [11]:
# A6 — Tổng hợp vấn đề đã quan sát trong raw_df (chưa sửa dữ liệu).
issues = []

# Thêm một vấn đề; tự đếm các dòng và lấy tối đa 5 record_id ví dụ.
def add_issue(field, issue, detection_rule, affected_rows):
    affected_rows = affected_rows.fillna(False)
    example_ids = raw_df.loc[affected_rows, 'record_id'].head(5)

    issues.append({
        'issue_id': f'ISSUE_{len(issues) + 1:03d}',
        'field': field,
        'issue': issue,
        'detection_rule': detection_rule,
        'affected_rows': int(affected_rows.sum()),
        'example_record_ids': '; '.join(example_ids.astype(str)),
    })

# Bỏ khoảng trắng hai đầu để khảo sát giá trị; raw_df không bị thay đổi.
raw_text = raw_df.astype('string')
raw_trimmed = raw_text.apply(lambda column: column.str.strip())
missing_tokens = [token.lower() for token in MISSING_TOKENS]

# Nhóm 1: dòng có ô rỗng hoặc ký hiệu thiếu ở các cột dữ liệu.
missing_cells = raw_trimmed.apply(
    lambda column: column.eq('') | column.str.lower().isin(missing_tokens)
)
missing_rows = missing_cells.drop(
    columns=['record_id', 'reservation_id']
).any(axis=1)

add_issue(
    'Nhiều cột',
    'Ô rỗng hoặc chứa ký hiệu thiếu',
    'Ô trống hoặc thuộc MISSING_TOKENS, bỏ qua hai cột ID',
    missing_rows,
)

# Nhóm 2: dòng trùng hoàn toàn trên 23 cột nghiệp vụ.
business_columns = [column for column in raw_df.columns if column != 'record_id']
exact_duplicate_rows = raw_df.duplicated(
    subset=business_columns,
    keep=False,
)

add_issue(
    '23 cột nghiệp vụ',
    'Dòng trùng tuyệt đối',
    'duplicated trên 23 cột, keep=False; bỏ record_id',
    exact_duplicate_rows,
)

# Nhóm 3: một reservation_id xuất hiện trên nhiều dòng.
reservation_key = raw_trimmed['reservation_id'].str.upper()
repeated_reservation_rows = reservation_key.duplicated(keep=False)

add_issue(
    'reservation_id',
    'Một reservation_id xuất hiện ở nhiều dòng',
    'reservation_id sau strip/upper bị lặp, keep=False',
    repeated_reservation_rows,
)

# Kiểm tra nhãn phân loại so với danh sách nhãn chuẩn của đề.
allowed_labels = {
    'airline': AIRLINES,
    'is_cancelled': ['0', '1'],
    'is_returning_customer': ['0', '1'],
    'meal_option': MEALS,
    'market_segment': SEGMENTS,
    'sales_channel': CHANNELS,
    'fare_policy': FARE_POLICIES,
    'traveler_type': TRAVELERS,
    'booking_status': STATUSES,
    'booked_cabin_code': CABIN_CODES,
    'assigned_cabin_code': CABIN_CODES,
}

for column, valid_labels in allowed_labels.items():
    # So sánh giá trị gốc để phát hiện cả khác biệt hoa/thường và khoảng trắng.
    values = raw_text[column]
    trimmed_values = raw_trimmed[column]
    has_value = trimmed_values.ne('') & ~trimmed_values.str.lower().isin(missing_tokens)

    # Mã cabin không phân biệt hoa/thường khi khảo sát.
    values_to_check = values

    nonstandard_rows = has_value & ~values_to_check.isin(valid_labels)

    add_issue(
        column,
        'Nhãn có biến thể hoặc không thuộc danh sách chuẩn',
        f'Giá trị không nằm trong nhãn chuẩn của {column}',
        nonstandard_rows,
    )

# Ngày khởi hành phải đọc được bằng một trong bốn format đã nêu ở B5.
departure_text = raw_trimmed['departure_date']
departure_has_value = departure_text.ne('') & ~departure_text.str.lower().isin(missing_tokens)
departure_parsed = pd.Series(pd.NaT, index=raw_df.index, dtype='datetime64[ns]')
date_formats = ['%Y-%m-%d', '%d/%m/%Y', '%B %d, %Y', '%d-%b-%Y']

for date_format in date_formats:
    parsed = pd.to_datetime(departure_text, format=date_format, errors='coerce')
    departure_parsed = departure_parsed.combine_first(parsed)

add_issue(
    'departure_date',
    'Có nhiều cách ghi ngày khởi hành',
    'Giá trị có dữ liệu nhưng không theo dạng ISO YYYY-MM-DD',
    departure_has_value & ~departure_text.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False),
)

add_issue(
    'departure_date',
    'Ngày sai định dạng hoặc không tồn tại',
    'Không đọc được bằng bốn format ngày quy định',
    departure_has_value & departure_parsed.isna(),
)

# Ngày trạng thái chỉ được ghi theo ISO YYYY-MM-DD.
status_date_text = raw_trimmed['booking_status_date']
status_date_has_value = status_date_text.ne('') & ~status_date_text.str.lower().isin(missing_tokens)
status_date_parsed = pd.to_datetime(
    status_date_text,
    format='%Y-%m-%d',
    errors='coerce',
)

add_issue(
    'booking_status_date',
    'Ngày không đúng ISO hoặc không tồn tại',
    "Không đọc được theo format '%Y-%m-%d'",
    status_date_has_value & status_date_parsed.isna(),
)

# Các cột đếm có thể chứa hậu tố đơn vị, số lẻ hoặc số âm.
integer_columns = [
    'booking_lead_days', 'trip_weekend_days', 'trip_weekdays',
    'adult_passengers', 'child_passengers', 'infant_passengers',
    'special_service_requests',
]
integer_text = raw_trimmed[integer_columns].apply(
    lambda column: column.str.lower()
)
integer_has_value = integer_text.apply(
    lambda column: column.ne('') & ~column.isin(missing_tokens)
)
integer_format_issue = integer_text.apply(
    lambda column: column.str.contains(r'[A-Za-z]|\.|-', regex=True, na=False)
)

add_issue(
    ', '.join(integer_columns),
    'Số có hậu tố đơn vị, phần thập phân hoặc dấu âm',
    'Tìm chữ, dấu chấm hoặc dấu âm trong các cột đếm',
    (integer_has_value & integer_format_issue).any(axis=1),
)

# Giá vé có thể ghi kèm tiền tệ hoặc dùng dấu phẩy thập phân.
fare_text = raw_trimmed['avg_ticket_fare'].str.lower()
fare_has_value = fare_text.ne('') & ~fare_text.isin(missing_tokens)
fare_format_issue = fare_text.str.contains(r'usd|\$|,|-', regex=True, na=False)

add_issue(
    'avg_ticket_fare',
    'Giá vé có ký hiệu tiền tệ, dấu phẩy hoặc dấu âm',
    'Tìm USD, $, dấu phẩy thập phân hoặc dấu âm',
    fare_has_value & fare_format_issue,
)

# Mã quốc gia có thể là tên đầy đủ, sai kiểu chữ hoặc sai định dạng.
country_text = raw_trimmed['passenger_country']
country_original = raw_text['passenger_country']
country_has_value = country_text.ne('') & ~country_text.str.lower().isin(missing_tokens)
country_is_code = country_text.str.upper().str.fullmatch(r'[A-Z]{2,3}', na=False)
country_issue_rows = country_has_value & (
    ~country_is_code
    | country_text.ne(country_text.str.upper())
    | country_original.ne(country_text)
)

add_issue(
    'passenger_country',
    'Tên quốc gia, chữ thường hoặc mã sai định dạng',
    'Sau strip, không phải mã 2–3 chữ in hoa',
    country_issue_rows,
)

# Tạo bảng theo đúng cấu trúc A6 và hiển thị kết quả.
issue_catalog = pd.DataFrame(
    issues,
    columns=[
        'issue_id', 'field', 'issue', 'detection_rule',
        'affected_rows', 'example_record_ids',
    ],
)

display(issue_catalog)


,issue_id,field,issue,detection_rule,affected_rows,example_record_ids
0,ISSUE_001,Nhiều cột,Ô rỗng hoặc chứa ký hiệu thiếu,"Ô trống hoặc thuộc MISSING_TOKENS, bỏ qua hai ...",301,R000068; R000118; R000173; R000180; R000193
1,ISSUE_002,23 cột nghiệp vụ,Dòng trùng tuyệt đối,"duplicated trên 23 cột, keep=False; bỏ record_id",360,R000042; R000072; R000114; R000117; R000132
2,ISSUE_003,reservation_id,Một reservation_id xuất hiện ở nhiều dòng,"reservation_id sau strip/upper bị lặp, keep=False",646,R000042; R000068; R000072; R000108; R000114
3,ISSUE_004,airline,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của airline,175,R000002; R000040; R000067; R000095; R000186
4,ISSUE_005,is_cancelled,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của is_canc...,123,R000161; R000331; R000332; R000485; R000493
5,ISSUE_006,is_returning_customer,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của is_retu...,43,R000096; R000437; R000716; R000996; R001005
6,ISSUE_007,meal_option,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của meal_op...,137,R000036; R000212; R000300; R000417; R000531
7,ISSUE_008,market_segment,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của market_...,153,R000052; R000143; R000236; R000250; R000326
8,ISSUE_009,sales_channel,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của sales_c...,102,R000120; R000149; R000305; R000370; R000548
9,ISSUE_010,fare_policy,Nhãn có biến thể hoặc không thuộc danh sách chuẩn,Giá trị không nằm trong nhãn chuẩn của fare_po...,95,R000136; R000587; R000606; R000748; R001014


In [12]:
# ✅ Tự kiểm tra giai đoạn A
check('A2 missing_survey có 24 dòng (1 dòng/cột)', lambda: len(missing_survey) == 24)
check('A2 missing_survey đủ cột', lambda: {'column','n_rows','empty_count','token_count','missing_total','missing_pct'} <= set(missing_survey.columns))
check('A6 issue_catalog có ≥ 10 nhóm vấn đề', lambda: len(issue_catalog) >= 10, 'Xem lại gợi ý trong thẻ nhiệm vụ A.')
check('A6 mỗi nhóm có ≤ 5 record_id ví dụ', lambda: issue_catalog['example_record_ids'].str.split('; ').str.len().le(5).all())
check('raw_df vẫn nguyên vẹn', lambda: raw_df.shape == (10325, 24))
save_table(missing_survey, 'a_missing_survey')
save_table(issue_catalog, 'a_issue_catalog')

✅ A2 missing_survey có 24 dòng (1 dòng/cột)
✅ A2 missing_survey đủ cột
✅ A6 issue_catalog có ≥ 10 nhóm vấn đề
✅ A6 mỗi nhóm có ≤ 5 record_id ví dụ
✅ raw_df vẫn nguyên vẹn
💾 outputs/tables/a_missing_survey.csv (24 dòng)
💾 outputs/tables/a_issue_catalog.csv (20 dòng)


> **✍️ Phát hiện chính và quy tắc dự kiến** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: chỉ xem các lỗi và tạo ra các bảng mới không sử dụng hay chỉnh sửa file csv dữ liệu
> - Vì sao chọn cách này thay vì cách khác:cách này sẽ dữ được bộ dữ liệu gốc không bị chỉnh sử
> - Ảnh hưởng đến kết quả phân tích phía sau: không ảnh hưởng đến kết quả phân tích phía sau, nó giúp hiểu bộ dữ liệu và hoàn thành các nhiệm vụ phía sau tốt hơn


---
# GIAI ĐOẠN B · LÀM SẠCH (B1 → B8)

**Nguyên tắc chung cho mọi bước**

1. Làm trên `df` (bản sao); **không bao giờ sửa `raw_df`**.
2. Mỗi quy tắc → **gọi `log_step(...)` một lần** với số liệu đếm được bằng mã, không gõ tay.
3. Giá trị không sửa được bằng bằng chứng có sẵn → **để thiếu (`pd.NA`)**, không đoán, không điền giá trị phổ biến nhất hay trung bình.
4. Đếm **trước và sau** mỗi bước để điền `check_after`.

| Cột nhật ký | Ý nghĩa ngắn |
|---|---|
| `affected_rows` | Số dòng **thỏa điều kiện phát hiện** |
| `changed_cells` | Số **ô thực sự đổi giá trị** |
| `removed_rows` | Số dòng **bị loại khỏi bảng** (thường chỉ khác 0 ở bước trùng/gộp) |

In [13]:
# Bản làm việc — chạy lại ô này nếu muốn làm lại giai đoạn B từ đầu
df = raw_df.copy()
cleaning_log_rows.clear()
print('df:', df.shape)
cleaning_log_rows

df: (10325, 24)


[]

## B1 · Khóa và bản ghi trùng tuyệt đối

| | |
|---|---|
| **Mục tiêu** | Phân biệt `record_id` (dòng nhập) với `reservation_id` (đặt chỗ); xử lý trùng tuyệt đối trên 23 cột nghiệp vụ. |
| **Đầu vào → Đầu ra** | `df` → `df` có `reservation_id` đã chuẩn hóa |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 7 · B1 |

**Các bước cần làm**

1. Chuẩn hóa `reservation_id`: bỏ khoảng trắng đầu/cuối, chuyển chữ hoa; dạng chuẩn là **`AR` + 6 chữ số**.
2. Đếm dòng trùng trên 23 cột nghiệp vụ; lưu ví dụ trước khi xử lý.
3. Có thể loại trùng tuyệt đối ngay hoặc để B8 gộp; phải ghi rõ trong nhật ký. Nếu loại ngay, giữ danh sách `record_id` để lập `row_lineage`.

**Gợi ý:** `str.strip().str.upper()`, `duplicated(subset=..., keep='first')`, `str.fullmatch(r'AR\d{6}')`

**⚠️ Bẫy thường gặp:** không được gộp hai reservation khác `reservation_id`; không được xóa dòng mà mất `record_id` truy vết.


In [14]:
# TODO B1

b1_changed = (df['reservation_id'] != df['reservation_id'].str.strip().str.upper()).sum()
df['reservation_id'] = df['reservation_id'].astype(str).str.strip().str.upper()

log_step(step='B1', 
         field='reservation_id', 
         issue='Dư khoảng trắng, chữ thường', 
         detection_rule='Khác AR + 6 chữ số', 
         action='strip, upper', 
         affected_rows=b1_changed, 
         changed_cells=b1_changed, 
         reason='Quy ước Mục 5.1', 
         check_after='100% AR+6 số')

In [15]:
check('B1 reservation_id đúng dạng AR + 6 chữ số', lambda: df['reservation_id'].str.fullmatch(r'AR\d{6}').all())


✅ B1 reservation_id đúng dạng AR + 6 chữ số


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: Chuẩn hóa reservation_id bằng cách bỏ khoảng trắng và viết hoa; so sánh trùng tuyệt đối trên 23 cột nghiệp vụ, đồng thời giữ nguyên các dòng và record_id để B8 xử lý.

> - Vì sao chọn cách này thay vì cách khác: Cách này bảo toàn dấu vết từng dòng nhập, tránh xóa dữ liệu trước khi lập row_lineage và không gộp nhầm các đặt chỗ có reservation_id khác nhau.
> - Ảnh hưởng đến kết quả phân tích phía sau: Cách này bảo toàn dấu vết từng dòng nhập, tránh xóa dữ liệu trước khi lập row_lineage và không gộp nhầm các đặt chỗ có reservation_id khác nhau.

## B2 · Giá trị nằm nhầm cột

| | |
|---|---|
| **Mục tiêu** | Chuyển ngày nằm nhầm ở `booking_status` sang `booking_status_date` **khi đủ bằng chứng**. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 20 phút |
| **Đọc kèm** | Đề bài Mục 7 · B2 |

**Các bước cần làm**

1. Tìm dòng thỏa **cả 2 điều kiện**: `booking_status` (sau strip) có dạng ngày `YYYY-MM-DD` **và** `booking_status_date` đang rỗng.
2. Chỉ với các dòng đó: chuyển giá trị sang `booking_status_date`, để `booking_status` rỗng, ghi nhật ký.
3. Làm bước này **trước** khi chuẩn hóa nhãn (B4): nếu ánh xạ nhãn trước, các ngày nằm nhầm sẽ bị biến thành thiếu và mất bằng chứng.

**Gợi ý hàm:** `str.strip()`, `str.fullmatch(r'\d{4}-\d{2}-\d{2}')`, toán tử `&`

**⚠️ Bẫy thường gặp**

- Ghi đè `booking_status_date` đang có dữ liệu.
- Đoán trạng thái cho các dòng vừa chuyển — trạng thái để thiếu.

In [16]:
# TODO B2

mask_b2 = df['booking_status'].astype(str).str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}') & (df['booking_status_date'].astype(str).str.strip() == '')
b2_count = mask_b2.sum()

# Chuyển dữ liệu sang đúng cột và trả cột cũ về NA
df.loc[mask_b2, 'booking_status_date'] = df.loc[mask_b2, 'booking_status'].str.strip()
df.loc[mask_b2, 'booking_status'] = pd.NA

log_step(step='B2', \
         field='booking_status', 
         issue='Ngày nằm nhầm ở trạng thái', 
         detection_rule='Khớp YYYY-MM-DD và ngày trạng thái trống', 
         action='Chuyển ngày sang booking_status_date, gán NA', 
         affected_rows=b2_count, 
         changed_cells=b2_count*2, 
         reason='Mục 7 B2', 
         check_after='Không còn ngày ở trạng thái')

In [17]:
check('B2 booking_status không còn chứa ngày', lambda: not df['booking_status'].astype(str).str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}').any())


✅ B2 booking_status không còn chứa ngày


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - **Quy tắc nhóm đã áp dụng:** Chỉ chuyển giá trị `YYYY-MM-DD` từ `booking_status` sang `booking_status_date` khi cột đích đang rỗng hoặc thiếu; sau đó để `booking_status` thiếu và ghi các dòng bị ảnh hưởng vào nhật ký.
> - **Vì sao chọn cách này thay vì cách khác:** Điều kiện cột đích trống giúp tránh ghi đè ngày đã có; nhóm không tự đoán trạng thái hay chọn giữa hai ngày khi cả hai cột đều có dữ liệu.
> - **Ảnh hưởng đến kết quả phân tích phía sau:** B4 không nhầm ngày với nhãn trạng thái, còn B5 sẽ kiểm tra ngày có tồn tại và nằm trong miền hợp lệ hay không; ngày lỗi sẽ thành `NaT`, không bị tự sửa.

## B3 · Giá trị thiếu theo từng cột

| | |
|---|---|
| **Mục tiêu** | Đưa các ký hiệu thiếu về `pd.NA`; giữ thiếu trong dữ liệu. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 7 · B3 |

**Các bước cần làm**

1. Ô rỗng `''` và `MISSING_TOKENS` (`NULL`, `N/A`, `NA`, `?`, `unknown`, `-`) → `pd.NA` ở mọi cột trừ `record_id`, `reservation_id`.
2. `Undefined` ở `market_segment`, `sales_channel` → thiếu (nguồn dùng nhãn này cho *không xác định*).
3. Ghi một dòng nhật ký cho mỗi cột có thay đổi (hoặc một dòng chung nhưng nêu đủ cột).

**Gợi ý hàm:** `str.lower().isin([t.lower() for t in MISSING_TOKENS])`, `mask`, `replace`

**⚠️ Bẫy thường gặp**

- Điền `child_passengers`, `passenger_country`, `avg_ticket_fare` bằng giá trị phổ biến/trung bình (bị cấm).
- Xóa mọi dòng có ô thiếu (bị cấm) — một đặt chỗ thiếu quốc gia vẫn dùng được khi tính tỷ lệ hủy theo hãng hàng không.

In [18]:
# TODO B3
# B3 · Giá trị thiếu theo từng cột
missing_tokens_lower = [t.lower() for t in MISSING_TOKENS]
cols_b3 = [c for c in df.columns if c not in ('record_id', 'reservation_id')]

b3_count = 0
for col in cols_b3:
    col_str = df[col].astype(str).str.strip().str.lower()
    mask_missing = col_str.isin(missing_tokens_lower) | (col_str == '')
    
    # Xử lý riêng cho 2 cột theo Mục 5.2
    if col in ['market_segment', 'sales_channel']:
        mask_missing = mask_missing | (col_str == 'undefined')
        
    b3_count += mask_missing.sum()
    df.loc[mask_missing, col] = pd.NA

log_step(step='B3', 
         field='Mọi cột (trừ ID)', 
         issue='Ký hiệu thiếu dạng chữ/rỗng', 
         detection_rule='Thuộc MISSING_TOKENS hoặc rỗng', 
         action='Gán pd.NA',
         affected_rows=b3_count, 
         changed_cells=b3_count,
         reason='Mục 5.1 và 7 B3', 
         check_after='Chỉ còn pd.NA cho ô rỗng')


In [19]:
check('B3 không còn ký hiệu thiếu dạng chữ',
      lambda: not any(df[c].dropna().astype(str).str.lower().isin([t.lower() for t in MISSING_TOKENS]).any()
                      for c in df.columns if c not in ('record_id', 'reservation_id')))


✅ B3 không còn ký hiệu thiếu dạng chữ


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - **Quy tắc nhóm đã áp dụng:** Chuyển ô rỗng và các token `NULL`, `N/A`, `NA`, `?`, `unknown`, `-` thành `pd.NA`; chỉ coi `Undefined` là thiếu trong `market_segment` và `sales_channel`.
> - **Vì sao chọn cách này thay vì cách khác:** Giá trị thiếu được giữ đúng là thiếu, không xóa cả dòng và không tự điền giá trị chưa có bằng chứng; `record_id` và `reservation_id` được giữ nguyên để truy vết.
> - **Ảnh hưởng đến kết quả phân tích phía sau:** Các token thiếu không bị tính nhầm thành nhãn hợp lệ; những đặt chỗ thiếu một thuộc tính vẫn được giữ để dùng cho các phân tích khác, còn số thiếu được phản ánh rõ trong báo cáo chất lượng.

## B4 · Chuẩn hóa nhãn phân loại và nhất quán trạng thái

| | |
|---|---|
| **Mục tiêu** | Đưa mọi cột phân loại về đúng tập nhãn chuẩn (Mục 5.2) và làm `is_cancelled` nhất quán với `booking_status` (Mục 5.3). |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 5.2, 5.3 và Mục 7 · B4 |

**Các bước cần làm**

1. Mọi cột (trừ `record_id`): `strip` + gom nhiều khoảng trắng thành 1.
2. Lập từ điển ánh xạ cho `airline, meal_option, market_segment, sales_channel, fare_policy, traveler_type, booking_status`. In `value_counts` **trước và sau**.
3. `is_cancelled`, `is_returning_customer`: đưa `yes/no`, `True/False`, `1.0/0.0` về số 0/1 (kiểu `Int64`).
4. `booked_cabin_code`, `assigned_cabin_code`: chữ hoa; mã ngoài tập hợp lệ → thiếu.
5. Đếm số dòng `is_cancelled` **mâu thuẫn** với `booking_status` (Flown ↔ 0; Cancelled, No-Show ↔ 1). Khi `booking_status` hợp lệ, **tính lại** `is_cancelled` từ nó; khi trạng thái thiếu, giữ `is_cancelled` đã chuẩn hóa. Ghi số dòng bị sửa.

**Gợi ý hàm:** `str.lower().str.replace(r'[^a-z]', '', regex=True)` để tạo khóa so khớp, `map(dict)`, `isin`

**⚠️ Bẫy thường gặp**

- `map` với từ điển thiếu khóa sẽ biến giá trị đó thành NaN — luôn kiểm tra số NaN mới phát sinh sau khi `map`.
- Đổi nhãn `Undefined` của `meal_option` thành thiếu — theo tài liệu nguồn, `Undefined` và `No Meal` cùng nghĩa *không kèm bữa ăn* (xem Mục 5.2).

In [20]:
# Từ điển: khóa là dạng sau clean_key(), giá trị là nhãn chuẩn.
AIRLINE_MAP = {'auroraair': 'Aurora Air', 
               'aurora': 'Aurora Air', 
               'horizonwings': 'Horizon Wings', 
               'horizon': 'Horizon Wings'}
MEAL_MAP = {'standard': 'Standard', 
            'std': 'Standard', 
            'standardmeal': 'Standard', 
            'vegetarian': 'Vegetarian', 'veg': 
            'Vegetarian', 'special': 'Special', 
            'specialmeal': 'Special', 
            'nomeal': 'No Meal', 
            'none': 'No Meal', 
            'undefined': 'No Meal'}
SEGMENT_MAP = {'direct': 'Direct', 
               'onlineota': 'Online OTA', 
               'onlinetravelagency': 'Online OTA', 
               'offlineagent': 'Offline Agent', 
               'corporate': 'Corporate', 
               'corp': 'Corporate', 
               'group': 'Group', 'groups': 'Group', 
               'packagetour': 'Package Tour', 'package': 
               'Package Tour', 'loyalty': 'Loyalty', 
               'loyaltyprogram': 'Loyalty'}
CHANNEL_MAP = {'website': 'Website', 
               'web': 'Website', 
               'mobileapp': 'Mobile App', 
               'app': 'Mobile App', 
               'travelagency': 'Travel Agency', 
               'ta': 'Travel Agency', 
               'gdscorporate': 'GDS/Corporate'}
FARE_POLICY_MAP = {'flexible': 'Flexible', 
                   'flex': 'Flexible', 
                   'semiflexible': 'Semi-Flexible', 
                   'nonrefundable': 'Non-Refundable', 
                   'nonref': 'Non-Refundable'}
TRAVELER_MAP = {'individual': 'Individual', 
                'linkedbooking': 'Linked Booking', 
                'corpcontract': 'Corporate Contract', 
                'corporatecontract': 'Corporate Contract', 
                'group': 'Group'}
STATUS_MAP = {'flown': 'Flown', 
              'completed': 'Flown', 
              'cancelled': 'Cancelled', 
              'canceled': 'Cancelled', 
              'noshow': 'No-Show'}

# 1. Bỏ khoảng trắng thừa, không sửa record_id.
before = df.drop(columns='record_id').copy()
for col in before.columns:
    df[col] = df[col].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
changed = before.fillna('').astype(str) != df[before.columns].fillna('').astype(str)
log_step('B4', 
         'Các cột trừ record_id', 
         'Khoảng trắng thừa', 
         'strip và gom khoảng trắng', 
         'Chuẩn hóa khoảng trắng', changed.any(axis=1).sum(), 
         changed.sum().sum(), 
         example_record_ids=df.loc[changed.any(axis=1), 'record_id'].head(5).tolist(), 
         reason='Chuẩn hóa trước khi so khớp nhãn',
         check_after=f'{int(changed.sum().sum())} ô được kiểm tra thay đổi')

# 2. Dùng clean_key để gom các cách viết về nhãn chuẩn.
category_maps = {'airline': AIRLINE_MAP, 
                 'meal_option': MEAL_MAP, 
                 'market_segment': SEGMENT_MAP, 
                 'sales_channel': CHANNEL_MAP, 
                 'fare_policy': FARE_POLICY_MAP, 
                 'traveler_type': TRAVELER_MAP, 
                 'booking_status': STATUS_MAP}
for col, mapping in category_maps.items():
    before = df[col].copy()
    df[col] = before.map(clean_key).map(mapping)
    unknown = before.notna() & df[col].isna()
    if unknown.any():
        print(col, 'chưa ánh xạ:', before[unknown].value_counts().to_dict())
    print('\n', col, 'sau chuẩn hóa:\n', df[col].value_counts(dropna=False))
    changed = before.fillna('').astype(str) != df[col].fillna('').astype(str)
    log_step('B4', 
             col, 
             'Biến thể nhãn', 
             'Tra từ điển bằng clean_key', 
             'Đổi về nhãn chuẩn; nhãn lạ để thiếu', 
             changed.sum(), 
             changed.sum(), 
             example_record_ids=df.loc[changed, 'record_id'].head(5).tolist(), 
             reason='Theo tập nhãn chuẩn Mục 5.2',
             check_after=f'{int(unknown.sum())} nhãn lạ được để thiếu')

# 3. Viết hoa mã khoang; mã ngoài CABIN_CODES thành thiếu.
for col in ['booked_cabin_code', 'assigned_cabin_code']:
    before = df[col].copy()
    df[col] = before.astype('string').str.upper()
    df[col] = df[col].where(df[col].isin(CABIN_CODES), pd.NA)
    changed = before.fillna('').astype(str) != df[col].fillna('').astype(str)
    log_step('B4', 
             col, 
             'Mã khoang không hợp lệ', 
             'So sánh với CABIN_CODES', 
             'Viết hoa mã; mã lạ để thiếu', 
             changed.sum(), 
             changed.sum(), 
             example_record_ids=df.loc[changed, 'record_id'].head(5).tolist(), 
             reason='Theo Mục 5.2',
             check_after=f'{int((df[col].notna() & ~df[col].isin(CABIN_CODES)).sum())} mã ngoài danh sách còn lại')

# 4. Chuẩn hóa cờ.
flag_map = {'1': 1, 
            '1.0': 1, 
            'yes': 1, 
            'true': 1, 
            '0': 0, 
            '0.0': 0, 
            'no': 0, 
            'false': 0}
for col in ['is_cancelled', 'is_returning_customer']:
    before = df[col].copy()
    df[col] = before.astype('string').str.lower().map(flag_map).astype('Int64')
    changed = before.fillna('').astype(str) != df[col].astype('string').fillna('').astype(str)
    log_step('B4', 
             col, 
             'Cờ chưa ở dạng 0/1', 
             'Tra từ điển flag_map', 
             'Đổi yes/true thành 1, no/false thành 0', 
             changed.sum(), changed.sum(), 
             example_record_ids=df.loc[changed, 'record_id'].head(5).tolist(), 
             reason='Dùng Int64 để cho phép giá trị thiếu',
             check_after=f'{int((df[col].notna() & ~df[col].isin([0, 1])).sum())} giá trị ngoài 0/1 còn lại')

# 5. Trạng thái hợp lệ quyết định cờ; trạng thái thiếu thì giữ cờ hiện có.
status_map = {'Flown': 0, 
              'Cancelled': 1, 
              'No-Show': 1}

expected = df['booking_status'].map(status_map)
valid = df['booking_status'].isin(STATUSES)
before = df['is_cancelled'].copy()
wrong = valid & df['is_cancelled'].ne(expected).fillna(True)
df.loc[valid, 'is_cancelled'] = expected[valid].astype('Int64')
changed = before.astype('string').fillna('') != df['is_cancelled'].astype('string').fillna('')
print('Số cờ thiếu hoặc mâu thuẫn được sửa:', wrong.sum())
log_step('B4', 
         'is_cancelled', 
         'Không khớp trạng thái', 
         'Flown=0; Cancelled/No-Show=1', 
         'Tính lại cờ khi trạng thái hợp lệ', 
         changed.sum(), 
         changed.sum(), 
         example_record_ids=df.loc[changed, 'record_id'].head(5).tolist(), 
         reason='Trạng thái thiếu thì giữ cờ đã chuẩn hóa',
         check_after=f'{int((valid & df["is_cancelled"].ne(expected).fillna(True)).sum())} cờ còn mâu thuẫn')
df




 airline sau chuẩn hóa:
 airline
Aurora Air       6740
Horizon Wings    3585
Name: count, dtype: int64

 meal_option sau chuẩn hóa:
 meal_option
Standard      6161
No Meal       1908
Vegetarian    1277
Special        954
NaN             25
Name: count, dtype: int64
market_segment chưa ánh xạ: {'Travel Agent': 9}

 market_segment sau chuẩn hóa:
 market_segment
Online OTA       4039
Direct           1892
Offline Agent    1656
Corporate         952
Package Tour      743
Group             602
Loyalty           432
NaN                 9
Name: count, dtype: int64

 sales_channel sau chuẩn hóa:
 sales_channel
Travel Agency    3610
Website          3274
Mobile App       2443
GDS/Corporate     998
Name: count, dtype: int64

 fare_policy sau chuẩn hóa:
 fare_policy
Semi-Flexible     3783
Flexible          3749
Non-Refundable    2793
Name: count, dtype: int64

 traveler_type sau chuẩn hóa:
 traveler_type
Individual            6627
Linked Booking        1911
Corporate Contract    1000
Group      

,record_id,reservation_id,airline,is_cancelled,booking_lead_days,departure_date,trip_weekend_days,trip_weekdays,adult_passengers,child_passengers,infant_passengers,meal_option,passenger_country,market_segment,sales_channel,is_returning_customer,booked_cabin_code,assigned_cabin_code,fare_policy,traveler_type,avg_ticket_fare,special_service_requests,booking_status,booking_status_date
0,R000001,AR002912,Aurora Air,0,19,2024-11-06,4,8,4,0,1,Special,VNM,Offline Agent,Travel Agency,0,F2,F2,Semi-Flexible,Individual,829.51,2,Flown,2024-11-07
1,R000002,AR008211,Aurora Air,0,0,15-Nov-2024,2,4,1,0,0,Standard,USA,Online OTA,Website,1,E1,E1,Non-Refundable,Individual,151.68,0,Flown,2024-11-17
2,R000003,AR005564,Aurora Air,0,42,12/12/2024,0,2,2,0,0,Standard,VNM,Corporate,Travel Agency,0,E4,E4,Flexible,Group,146.28,1,Flown,2024-12-12
3,R000004,AR007528,Aurora Air,1,89,2024-08-20,0,4,1,0,0,Special,SWE,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,206.48,0,Cancelled,2024-06-07
4,R000005,AR007195,Aurora Air,0,18,10/02/2025,0,5,1,1,0,No Meal,GBR,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,186.89,1,Flown,2025-02-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10320,R010321,AR001894,Horizon Wings,0,85,2025-11-18,0,2,4,0,0,Vegetarian,KOR,Group,Travel Agency,0,E1,E1,Semi-Flexible,Group,100.29,0,Flown,2025-11-18
10321,R010322,AR009560,Aurora Air,0,30,2025-06-29,1,2,2,0,0,Vegetarian,KOR,Package Tour,Travel Agency,0,E2,P2,Non-Refundable,Linked Booking,131.7,0,Flown,2025-06-30
10322,R010323,AR002073,Aurora Air,0,97,21/07/2024,1,5,3,0,0,Standard,AUT,Direct,Website,1,E3,E3,Semi-Flexible,Linked Booking,112.0,1,Flown,2024-07-23
10323,R010324,AR004057,Aurora Air,0,22.0,31/07/2025,2,3,2,0,0,Special,KOR,Direct,Website,0,P2,P2,Flexible,Individual,355.34,1,Flown,2025-08-01


In [21]:
check('B4 airline chỉ còn Aurora Air / Horizon Wings và không thiếu', lambda: set(df['airline']) <= set(AIRLINES) and df['airline'].notna().all())
check('B4 meal_option hợp lệ hoặc thiếu', lambda: df['meal_option'].dropna().isin(MEALS).all())
check('B4 market_segment hợp lệ hoặc thiếu', lambda: df['market_segment'].dropna().isin(SEGMENTS).all())
check('B4 sales_channel hợp lệ hoặc thiếu', lambda: df['sales_channel'].dropna().isin(CHANNELS).all())
check('B4 fare_policy hợp lệ hoặc thiếu', lambda: df['fare_policy'].dropna().isin(FARE_POLICIES).all())
check('B4 traveler_type hợp lệ hoặc thiếu', lambda: df['traveler_type'].dropna().isin(TRAVELERS).all())
check('B4 booking_status hợp lệ hoặc thiếu', lambda: df['booking_status'].dropna().isin(STATUSES).all())
check('B4 cabin code hợp lệ hoặc thiếu', lambda: df['booked_cabin_code'].dropna().isin(CABIN_CODES).all() and df['assigned_cabin_code'].dropna().isin(CABIN_CODES).all())
check('B4 hai cờ 0/1 kiểu Int64', lambda: str(df['is_cancelled'].dtype) == 'Int64' and str(df['is_returning_customer'].dtype) == 'Int64')


✅ B4 airline chỉ còn Aurora Air / Horizon Wings và không thiếu
✅ B4 meal_option hợp lệ hoặc thiếu
✅ B4 market_segment hợp lệ hoặc thiếu
✅ B4 sales_channel hợp lệ hoặc thiếu
✅ B4 fare_policy hợp lệ hoặc thiếu
✅ B4 traveler_type hợp lệ hoặc thiếu
✅ B4 booking_status hợp lệ hoặc thiếu
✅ B4 cabin code hợp lệ hoặc thiếu
✅ B4 hai cờ 0/1 kiểu Int64


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
Quy tắc nhóm đã áp dụng: Sử dụng Dictionary để map toàn bộ biến thể về nhãn chuẩn. Với mâu thuẫn trạng thái, nhóm dùng booking_status làm gốc để tính toán lại cờ is_cancelled (Flown -> 0; Cancelled/No-Show -> 1).

Vì sao chọn cách này thay vì cách khác: Trạng thái ghi nhận cuối cùng (booking_status) mang tính cập nhật và chi tiết hơn so với cột cờ boolean 0/1 đơn giản, do đó nó đáng tin cậy hơn để dùng làm mỏ neo đồng bộ dữ liệu.

Ảnh hưởng đến kết quả phân tích phía sau: Ngăn chặn sai số nghiêm trọng khi tính Tỷ lệ hủy (Cancel Rate). Nếu không đồng bộ, biểu đồ so sánh Hủy/Bay ở Câu 7 sẽ bị lệch.

## B5 · Ngày khởi hành và ngày cập nhật trạng thái

| | |
|---|---|
| **Mục tiêu** | Chuyển `departure_date` (4 định dạng; dạng `/` đọc ngày/tháng/năm) và `booking_status_date` sang kiểu ngày, đúng miền. |
| **Đầu vào → Đầu ra** | `df` → `df` với 2 cột datetime |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 5.1 và Mục 7 · B5 |

**Các bước cần làm**

1. Đọc từng định dạng của `departure_date` bằng `pd.to_datetime(..., format=..., errors='coerce')`, rồi ghép kết quả.
2. `departure_date` ngoài **01/07/2024 – 31/08/2026**, ngày không tồn tại hoặc không đọc được → `NaT`.
3. `booking_status_date` dùng ISO `YYYY-MM-DD`; ngoài **01/01/2024 – 15/09/2026** hoặc không tồn tại → `NaT`.
4. Không tự sửa ngày sai thành ngày gần nhất.

**Bốn format của `departure_date`:** `%Y-%m-%d`, `%d/%m/%Y`, `%B %d, %Y`, `%d-%b-%Y`.

**⚠️ Bẫy:** không để pandas tự đoán `05/03/2025`; phải xử lý B2 trước để không mất ngày nằm nhầm cột.


In [22]:
# Các định dạng ngày được nêu trong đề.
DATE_FORMATS = ['%Y-%m-%d', '%d/%m/%Y', '%B %d, %Y', '%d-%b-%Y']

# PHẦN 1 — Đọc departure_date theo từng định dạng.
departure_before = df['departure_date'].copy()
departure_date = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')

for date_format in DATE_FORMATS:
    parsed_date = pd.to_datetime(
        departure_before,
        format=date_format,
        errors='coerce'
    )
    # Giữ ngày đã đọc được; chỉ lấy parsed_date nếu ô hiện tại còn thiếu.
    departure_date = departure_date.combine_first(parsed_date)

# Ngày sai hoặc nằm ngoài khoảng đề cho phép sẽ thành NaT.
departure_in_range = departure_date.between(DEPARTURE_MIN, DEPARTURE_MAX)
departure_bad_count = (departure_before.notna() & ~departure_in_range).sum()
df['departure_date'] = departure_date.where(departure_in_range, pd.NaT)

print('departure_date không đọc được hoặc ngoài miền:', int(departure_bad_count))

departure_bad = departure_before.notna() & ~departure_in_range
log_step(
    step='B5',
    field='departure_date',
    issue='Nhiều định dạng ngày hoặc ngày ngoài miền',
    detection_rule='Thử lần lượt 4 format; kiểm tra miền ngày',
    action='Chuyển sang datetime; ngày sai/ngoài miền thành NaT',
    affected_rows=int(departure_before.notna().sum()),
    changed_cells=int(departure_before.notna().sum()),
    example_record_ids=df.loc[departure_bad, 'record_id'].head(5).tolist(),
    reason='Dạng / đọc theo ngày/tháng/năm; không tự sửa ngày sai',
    check_after=f'{int(departure_bad_count)} giá trị không đọc được hoặc ngoài miền'
)

# PHẦN 2 — booking_status_date chỉ dùng định dạng ISO YYYY-MM-DD.
status_date_before = df['booking_status_date'].copy()
status_date = pd.to_datetime(
    status_date_before,
    format='%Y-%m-%d',
    errors='coerce'
)

# Ngày sai hoặc nằm ngoài khoảng đề cho phép sẽ thành NaT.
status_in_range = status_date.between(STATUS_MIN, STATUS_MAX)
status_bad_count = (status_date_before.notna() & ~status_in_range).sum()
df['booking_status_date'] = status_date.where(status_in_range, pd.NaT)

print('booking_status_date không đọc được hoặc ngoài miền:', int(status_bad_count))

status_bad = status_date_before.notna() & ~status_in_range
log_step(
    step='B5',
    field='booking_status_date',
    issue='Ngày không đúng ISO hoặc ngoài miền',
    detection_rule='Đọc YYYY-MM-DD; kiểm tra miền ngày',
    action='Chuyển sang datetime; ngày sai/ngoài miền thành NaT',
    affected_rows=int(status_date_before.notna().sum()),
    changed_cells=int(status_date_before.notna().sum()),
    example_record_ids=df.loc[status_bad, 'record_id'].head(5).tolist(),
    reason='Theo miền ngày đề bài; không tự sửa ngày sai',
    check_after=f'{int(status_bad_count)} giá trị không đọc được hoặc ngoài miền'
)


departure_date không đọc được hoặc ngoài miền: 10
booking_status_date không đọc được hoặc ngoài miền: 8


In [23]:
check('B5 departure_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['departure_date'])
      and df['departure_date'].dropna().between(DEPARTURE_MIN, DEPARTURE_MAX).all())
check('B5 booking_status_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['booking_status_date'])
      and df['booking_status_date'].dropna().between(STATUS_MIN, STATUS_MAX).all())


✅ B5 departure_date là kiểu datetime, trong miền
✅ B5 booking_status_date là kiểu datetime, trong miền


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
Quy tắc nhóm đã áp dụng: Ép kiểu qua 4 định dạng ngày cụ thể bằng pd.to_datetime. Các ngày không hợp lệ hoặc nằm ngoài mốc 2024-2026 bị đổi thành NaT.

Vì sao chọn cách này thay vì cách khác: Việc khai báo rõ format (như %d/%m/%Y) tránh việc Pandas tự đoán sai ngày và tháng (ví dụ 05/03 bị hiểu nhầm thành mùng 3 tháng 5). Không tự ý sửa ngày sai (ví dụ 31/02) thành ngày hợp lệ gần nhất vì vi phạm quy tắc không đoán dữ liệu.

Ảnh hưởng đến kết quả phân tích phía sau: Đảm bảo độ tin cậy tuyệt đối cho cột dẫn xuất departure_month để vẽ đường xu hướng mùa vụ trong Câu 3 một cách chính xác.

## B6 · Số đo: lead time, số ngày chuyến đi, số hành khách, yêu cầu đặc biệt và giá vé

| | |
|---|---|
| **Mục tiêu** | Chuyển cột số sang kiểu đúng (`Int64` cho số nguyên, `float` cho `avg_ticket_fare`) và đúng miền. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 5.1 và Mục 7 · B6 |

**Các bước cần làm**

1. Miền số nguyên: `booking_lead_days` 0–365; `trip_weekend_days` 0–20; `trip_weekdays` 0–50; ba cột hành khách 0–10; `special_service_requests` 0–5.
2. Chấp nhận cách ghi như `2.0` → 2 và hậu tố đơn vị đúng nghĩa cột (ví dụ `3 days`); giá trị âm, không phải số hoặc ngoài miền → thiếu + ghi lý do.
3. `avg_ticket_fare` là **USD trên mỗi hành khách**: bỏ `$`, `USD`; đổi dấu phẩy thập phân `95,50` → `95.50`; miền 0–5000. **Giá 0 có thể hợp lệ**, không tự xóa.
4. Giá trị hiếm nhưng vẫn trong miền phải được giữ lại; outlier không đồng nghĩa với lỗi.

**Gợi ý:** `str.replace`, `pd.to_numeric(errors='coerce')`, `where`, `between`, `.astype('Int64')`

**⚠️ Bẫy:** dùng `int` thường khi có thiếu; xóa `avg_ticket_fare = 0`; hoặc biến `95,50` thành 9550.


In [24]:
# Giới hạn hợp lệ của từng cột số nguyên (theo đề bài).
INT_RULES = {
    'booking_lead_days': (0, 365),
    'trip_weekend_days': (0, 20),
    'trip_weekdays': (0, 50),
    'adult_passengers': (0, 10),
    'child_passengers': (0, 10),
    'infant_passengers': (0, 10),
    'special_service_requests': (0, 5),
}

# PHẦN 1 — Chuyển các cột số nguyên.
# Chỉ bỏ hậu tố đúng với cột đó, ví dụ '3 days' hoặc '2 pax'.
SUFFIX_PATTERNS = {
    'booking_lead_days': r'\s*days?$',
    'trip_weekend_days': r'\s*days?$',
    'trip_weekdays': r'\s*days?$',
    'adult_passengers': r'\s*pax$',
    'child_passengers': r'\s*child$',
    'infant_passengers': r'\s*infant$',
    'special_service_requests': r'\s*requests?$',
}

for col, (minimum, maximum) in INT_RULES.items():
    # Lưu bản gốc để tính số dòng bị tác động và làm ví dụ trong nhật ký.
    original = df[col].copy()

    # Đưa về chữ thường, bỏ khoảng trắng hai đầu và hậu tố đơn vị phù hợp.
    number_text = original.astype('string').str.strip().str.lower()
    number_text = number_text.str.replace(
        SUFFIX_PATTERNS[col],
        '',
        regex=True
    )

    # Chuyển chuỗi thành số; chữ hoặc ký hiệu lạ không đọc được thành thiếu.
    number = pd.to_numeric(number_text, errors='coerce')

    # Chỉ nhận số nguyên nằm trong miền cho phép.
    in_range = number.between(minimum, maximum)
    is_integer = number.mod(1).eq(0)
    valid_number = in_range & is_integer
    invalid_number = original.notna() & ~valid_number

    # Giá trị âm, lẻ, không phải số hoặc ngoài miền được chuyển thành thiếu.
    df[col] = number.where(valid_number, pd.NA).astype('Int64')

    print(f'{col}: {int(invalid_number.sum())} giá trị không hợp lệ thành thiếu')

    # Ghi số liệu và một số record_id ví dụ vào nhật ký làm sạch.
    log_step(
        step='B6',
        field=col,
        issue='Giá trị không phải số nguyên hợp lệ',
        detection_rule=f'Số nguyên trong khoảng {minimum} đến {maximum}',
        action='Bỏ hậu tố đúng nghĩa cột; chuyển số; giá trị sai thành thiếu',
        affected_rows=int(original.notna().sum()),
        changed_cells=int(original.notna().sum()),
        example_record_ids=df.loc[invalid_number, 'record_id'].head(5).tolist(),
        reason='Không đoán hoặc tự điền giá trị không hợp lệ',
        check_after=f'{int(invalid_number.sum())} giá trị bị chuyển thành thiếu'
    )

# PHẦN 2 — Làm sạch giá vé trung bình (USD / hành khách).
fare_before = df['avg_ticket_fare'].copy()

# Bỏ ký hiệu tiền tệ; dấu phẩy được hiểu là dấu thập phân, ví dụ 95,50 thành 95.50.
fare_text = fare_before.astype('string').str.lower().str.strip()
fare_text = fare_text.str.replace('usd', '', regex=False)
fare_text = fare_text.str.replace('$', '', regex=False)
fare_text = fare_text.str.replace(',', '.', regex=False).str.strip()

# Chuyển sang số và chỉ giữ giá trị từ 0 đến 5000.
fare_number = pd.to_numeric(fare_text, errors='coerce')
fare_in_range = fare_number.between(FARE_MIN, FARE_MAX)
fare_invalid = fare_before.notna() & ~fare_in_range
df['avg_ticket_fare'] = fare_number.where(fare_in_range)

# Giá 0 hợp lệ theo đề, nên vẫn được giữ lại.
print('avg_ticket_fare: giá 0 được giữ =', int(df['avg_ticket_fare'].eq(0).sum()))
print('avg_ticket_fare: giá không hợp lệ thành thiếu =', int(fare_invalid.sum()))

log_step(
    step='B6',
    field='avg_ticket_fare',
    issue='Giá vé không đọc được hoặc ngoài miền',
    detection_rule='Bỏ USD/$; đổi dấu phẩy thập phân; kiểm tra 0 đến 5000',
    action='Chuyển sang số; giá sai/ngoài miền thành thiếu; giữ giá 0',
    affected_rows=int(fare_before.notna().sum()),
    changed_cells=int(fare_before.notna().sum()),
    example_record_ids=df.loc[fare_invalid, 'record_id'].head(5).tolist(),
    reason='Giá vé là USD trên mỗi hành khách; giá 0 được đề xác nhận hợp lệ',
    check_after=f'{int(fare_invalid.sum())} giá trị bị chuyển thành thiếu'
)


booking_lead_days: 22 giá trị không hợp lệ thành thiếu
trip_weekend_days: 10 giá trị không hợp lệ thành thiếu
trip_weekdays: 9 giá trị không hợp lệ thành thiếu
adult_passengers: 7 giá trị không hợp lệ thành thiếu
child_passengers: 8 giá trị không hợp lệ thành thiếu
infant_passengers: 5 giá trị không hợp lệ thành thiếu
special_service_requests: 6 giá trị không hợp lệ thành thiếu
avg_ticket_fare: giá 0 được giữ = 13
avg_ticket_fare: giá không hợp lệ thành thiếu = 13


In [25]:
check('B6 các cột số nguyên có kiểu Int64 và đúng miền',
      lambda: all(str(df[c].dtype) == 'Int64' and df[c].dropna().between(lo, hi).all() for c, lo, hi in [
          ('booking_lead_days', LEAD_MIN, LEAD_MAX), ('trip_weekend_days', 0, WEEKEND_MAX),
          ('trip_weekdays', 0, WEEK_MAX), ('adult_passengers', 0, PASSENGER_MAX),
          ('child_passengers', 0, PASSENGER_MAX), ('infant_passengers', 0, PASSENGER_MAX),
          ('special_service_requests', 0, REQ_MAX)]))
check('B6 avg_ticket_fare là số trong 0–5000 hoặc thiếu', lambda: pd.api.types.is_numeric_dtype(df['avg_ticket_fare'])
      and df['avg_ticket_fare'].dropna().between(FARE_MIN, FARE_MAX).all())


✅ B6 các cột số nguyên có kiểu Int64 và đúng miền
✅ B6 avg_ticket_fare là số trong 0–5000 hoặc thiếu


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Nhóm đã bỏ hậu tố đơn vị phù hợp, chuyển các cột đếm về số nguyên `Int64`, và chuyển giá vé về số trong miền đề bài.
> - Giá trị âm, không đọc được, không nguyên ở cột đếm hoặc ngoài miền được để thiếu; nhóm không tự điền. Dấu phẩy trong giá vé được hiểu là dấu thập phân và giá `0` được giữ vì đề xác nhận có thể hợp lệ.
> - Các cột số sau làm sạch dùng được cho thống kê và phân tích; số giá trị không hợp lệ cùng ví dụ `record_id` đã được ghi trong nhật ký B6.

## B7 · Mã quốc gia của hành khách/đại diện đặt chỗ

| | |
|---|---|
| **Mục tiêu** | `passenger_country` là mã 2–3 chữ cái in hoa; tên đầy đủ được đổi về mã tương ứng. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 20 phút |
| **Đọc kèm** | Đề bài Mục 5.1 và Mục 7 · B7 |

**Các bước cần làm**

1. Bỏ khoảng trắng, viết hoa các mã đã là mã.
2. Ánh xạ cả giá trị cho các tên đầy đủ xuất hiện trong dữ liệu, ví dụ: `Vietnam → VNM`, `China → CHN`, `Japan → JPN`, `South Korea → KOR`, `Thailand → THA`, `United States → USA`, `United Kingdom → GBR`, `Australia → AUS`.
3. Giá trị sau chuẩn hóa không khớp `[A-Z]{2,3}` → thiếu + ghi nhật ký.
4. Giữ `CN` và `CHN` là hai mã khác nhau trong phạm vi bài nếu không có bằng chứng để gộp; nêu giới hạn nếu ảnh hưởng Câu 5.

**⚠️ Bẫy:** không thay chuỗi con; không đoán quốc gia từ hãng, kênh bán hoặc tên khác.


In [26]:
COUNTRY_MAP = {'VIETNAM': 'VNM', 'CHINA': 'CHN', 'JAPAN': 'JPN', 'SOUTH KOREA': 'KOR', 
    'THAILAND': 'THA', 'UNITED STATES': 'USA', 'UNITED KINGDOM': 'GBR', 'AUSTRALIA': 'AUS',
    'INDONESIA': 'IDN', 'SPAIN': 'ESP', 'SINGAPORE': 'SGP', 'TURKEY': 'TUR',
    'NETHERLANDS': 'NLD', 'MEXICO': 'MEX', 'MALAYSIA': 'MYS', 'ITALY': 'ITA',
    'EGYPT': 'EGY', 'GERMANY': 'DEU', 'INDIA': 'IND', 'CANADA': 'CAN'
    # TODO: 'VIETNAM': 'VNM', 'UNITED STATES': 'USA', ...
}
# TODO B7: strip, upper, map tên đầy đủ -> mã, mã ngoài 2–3 chữ A-Z -> thiếu; log_step

mask_c_notna = df['passenger_country'].notna()
country = df.loc[mask_c_notna, 'passenger_country'].astype(str).str.strip().str.upper().replace(COUNTRY_MAP)
mask_valid_c = country.str.fullmatch(r'[A-Z]{2,3}', na=False)
invalid_c_count = int((~mask_valid_c).sum())

df.loc[mask_c_notna, 'passenger_country'] = country.where(mask_valid_c, pd.NA)

log_step(step='B7', 
         field='passenger_country', 
         issue='Tên đầy đủ hoặc mã lỗi', 
         detection_rule='Độ dài > 3 hoặc chứa ký tự đặc biệt', 
         action='Map về mã 2-3 chữ, gán NA', 
         affected_rows=invalid_c_count, 
         changed_cells=invalid_c_count,
         reason='Mục 5.1', 
         check_after='Mã in hoa 2-3 chữ hoặc NA')

In [27]:
check('B7 passenger_country là mã 2–3 chữ in hoa hoặc thiếu', lambda: df['passenger_country'].dropna().str.fullmatch(r'[A-Z]{2,3}').all())


✅ B7 passenger_country là mã 2–3 chữ in hoa hoặc thiếu


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - **Quy tắc nhóm đã chọn:** Bỏ khoảng trắng, viết hoa giá trị `passenger_country`, ánh xạ toàn bộ tên quốc gia đã biết sang mã tương ứng; giá trị còn lại không khớp `[A-Z]{2,3}` sẽ để thiếu và ghi nhật ký.
> - **Vì sao chọn cách này thay vì cách khác:** Ánh xạ toàn giá trị giúp tránh đoán quốc gia từ một phần chuỗi; nhóm giữ `CN` và `CHN` riêng vì chưa có bằng chứng để coi chúng là một mã.
> - **Ảnh hưởng đến kết quả phân tích phía sau:** Mã hợp lệ giúp đếm và xếp hạng quốc gia nhất quán ở Câu 5; giá trị không xác định sẽ không bị gán nhầm quốc gia, còn việc giữ riêng `CN`/`CHN` có thể làm phân tán số lượng giữa hai mã.

## B8 · Hợp nhất các dòng cùng reservation_id và lập truy vết

| | |
|---|---|
| **Mục tiêu** | Mỗi `reservation_id` còn **đúng 1 dòng**; mọi `record_id` gốc truy được về `reservation_id` cuối cùng. |
| **Đầu vào → Đầu ra** | `df` → `merged_df`, `row_lineage` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 7 · B8 |

**Các bước cần làm**

1. Với các dòng cùng `reservation_id`, so sánh **giá trị không thiếu** theo từng cột.
2. Giá trị đồng thuận → giữ. Một bên thiếu, một bên có → lấy giá trị có. **Mâu thuẫn** → áp dụng quy tắc nhóm tự chọn (ví dụ: lấy từ dòng có `record_id` nhỏ nhất) và lưu danh sách xung đột.
3. Tạo `row_lineage` với cột `record_id, reservation_id, status` (`kept`, `merged`, `dropped`).
4. Ghi nhật ký: số dòng trước, số dòng sau, số dòng loại, số reservation_id có xung đột.

**Gợi ý hàm:** `groupby('reservation_id')`, `nunique(dropna=True)`, `first()` (bỏ qua NaN), `duplicated`

**⚠️ Bẫy thường gặp**

- Gộp trước khi chuẩn hóa xong → bỏ sót trùng hoặc tạo xung đột giả (ví dụ `95,50` và `95.5`).
- Quên các dòng đã loại ở B1 trong `row_lineage`.

In [28]:
# PHẦN 1 — Sắp xếp để quy tắc chọn dòng luôn ổn định.
# Khi có mâu thuẫn, dòng có record_id nhỏ nhất sẽ được ưu tiên.
df_b8 = df.sort_values('record_id').copy()

# PHẦN 2 — Tạo danh sách xung đột trước khi gộp.
# Chỉ coi các giá trị không thiếu là xung đột.
conflict_rows = []

for reservation_id, group in df_b8.groupby('reservation_id', sort=False):
    for column in df_b8.columns:
        if column in ['record_id', 'reservation_id']:
            continue

        values = group[column].dropna().drop_duplicates()

        if len(values) > 1:
            conflict_rows.append({
                'reservation_id': reservation_id,
                'column': column,
                'values': values.tolist(),
                'record_ids': group.loc[group[column].notna(), 'record_id'].tolist(),
            })

# Bảng này giữ lại chi tiết để nhóm xem xét các xung đột.
b8_conflicts = pd.DataFrame(
    conflict_rows,
    columns=['reservation_id', 'column', 'values', 'record_ids']
)

# PHẦN 3 — Gộp các dòng cùng reservation_id.
# groupby().first() lấy giá trị không thiếu đầu tiên của từng cột.
# Do đã sắp xếp theo record_id, giá trị đầu tiên là quy tắc ưu tiên đã chọn.
merged_df = (
    df_b8
    .drop(columns='record_id')
    .groupby('reservation_id', as_index=False, sort=False)
    .first()
)

# PHẦN 4 — Tạo nhãn truy vết cho các dòng còn trong df.
# Dòng đầu tiên của mỗi reservation được giữ; các dòng còn lại đã được gộp.
row_lineage = df_b8[['record_id', 'reservation_id']].copy()
row_lineage['status'] = 'merged'
first_rows = ~df_b8.duplicated('reservation_id', keep='first')
row_lineage.loc[first_rows, 'status'] = 'kept'

# Nếu bước B1 trước đó đã loại dòng, vẫn thêm các record_id đó vào truy vết.
lineage_status = row_lineage.set_index('record_id')['status']
row_lineage = raw_df[['record_id', 'reservation_id']].copy()
row_lineage['reservation_id'] = (
    row_lineage['reservation_id'].astype('string').str.strip().str.upper()
)
row_lineage['status'] = row_lineage['record_id'].map(lineage_status).fillna('dropped')

# PHẦN 5 — Báo cáo kích thước và ghi nhật ký B8.
rows_before = len(df)
rows_after = len(merged_df)
rows_merged = rows_before - rows_after
conflict_reservations = b8_conflicts['reservation_id'].nunique()

print('Số dòng trước khi gộp:', rows_before)
print('Số reservation sau khi gộp:', rows_after)
print('Số dòng được gộp vào reservation khác:', rows_merged)
print('Số reservation có xung đột:', conflict_reservations)
print('\nMột số xung đột cần xem xét:')
print(b8_conflicts.head(10).to_string(index=False))

log_step(
    step='B8',
    field='reservation_id và các cột nghiệp vụ',
    issue='Nhiều dòng cho cùng reservation_id',
    detection_rule='Group theo reservation_id; xung đột khi có hơn một giá trị không thiếu',
    action='Sắp record_id tăng dần; lấy giá trị không thiếu đầu tiên; ghi bảng xung đột',
    affected_rows=rows_before,
    changed_cells=0,
    removed_rows=rows_merged,
    example_record_ids=df_b8.loc[~first_rows, 'record_id'].head(5).tolist(),
    reason='Giá trị thiếu được bổ sung từ dòng khác; khi mâu thuẫn ưu tiên record_id nhỏ nhất',
    check_after=f'{rows_after} reservation duy nhất; {conflict_reservations} reservation có xung đột'
)
display(merged_df)

Số dòng trước khi gộp: 10325
Số reservation sau khi gộp: 10000
Số dòng được gộp vào reservation khác: 325
Số reservation có xung đột: 85

Một số xung đột cần xem xét:
reservation_id          column                   values                  record_ids
      AR005030     meal_option      [No Meal, Standard]          [R000108, R008293]
      AR000698     meal_option    [Vegetarian, Special] [R000139, R002124, R009755]
      AR000698   sales_channel [Travel Agency, Website] [R000139, R002124, R009755]
      AR003643 avg_ticket_fare           [263.0, 258.0]          [R000140, R003558]
      AR004530 avg_ticket_fare         [106.18, 111.18]          [R000185, R007801]
      AR006207     meal_option      [Standard, Special]          [R000200, R004899]
      AR001318     meal_option      [Special, Standard]          [R000224, R006268]
      AR009713 avg_ticket_fare         [155.83, 150.83]          [R000232, R008072]
      AR007831 avg_ticket_fare         [171.45, 176.45]          [R000282, R0

,reservation_id,airline,is_cancelled,booking_lead_days,departure_date,trip_weekend_days,trip_weekdays,adult_passengers,child_passengers,infant_passengers,meal_option,passenger_country,market_segment,sales_channel,is_returning_customer,booked_cabin_code,assigned_cabin_code,fare_policy,traveler_type,avg_ticket_fare,special_service_requests,booking_status,booking_status_date
0,AR002912,Aurora Air,0,19,2024-11-06,4,8,4,0,1,Special,VNM,Offline Agent,Travel Agency,0,F2,F2,Semi-Flexible,Individual,829.51,2,Flown,2024-11-07
1,AR008211,Aurora Air,0,0,2024-11-15,2,4,1,0,0,Standard,USA,Online OTA,Website,1,E1,E1,Non-Refundable,Individual,151.68,0,Flown,2024-11-17
2,AR005564,Aurora Air,0,42,2024-12-12,0,2,2,0,0,Standard,VNM,Corporate,Travel Agency,0,E4,E4,Flexible,Group,146.28,1,Flown,2024-12-12
3,AR007528,Aurora Air,1,89,2024-08-20,0,4,1,0,0,Special,SWE,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,206.48,0,Cancelled,2024-06-07
4,AR007195,Aurora Air,0,18,2025-02-10,0,5,1,1,0,No Meal,GBR,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,186.89,1,Flown,2025-02-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,AR006752,Aurora Air,0,256,2026-02-17,0,1,1,0,0,Standard,SGP,Online OTA,Mobile App,0,E3,E3,Flexible,Individual,85.49,0,Flown,2026-02-17
9996,AR001894,Horizon Wings,0,85,2025-11-18,0,2,4,0,0,Vegetarian,KOR,Group,Travel Agency,0,E1,E1,Semi-Flexible,Group,100.29,0,Flown,2025-11-18
9997,AR002073,Aurora Air,0,97,2024-07-21,1,5,3,0,0,Standard,AUT,Direct,Website,1,E3,E3,Semi-Flexible,Linked Booking,112.0,1,Flown,2024-07-23
9998,AR004057,Aurora Air,0,22,2025-07-31,2,3,2,0,0,Special,KOR,Direct,Website,0,P2,P2,Flexible,Individual,355.34,1,Flown,2025-08-01


In [29]:
check('B8 merged_df: reservation_id duy nhất', lambda: merged_df['reservation_id'].is_unique)
check('B8 row_lineage phủ đủ 10.325 record_id', lambda: set(row_lineage['record_id']) == set(raw_df['record_id']))
check('B8 số dòng kept = số reservation', lambda: (row_lineage['status'] == 'kept').sum() == len(merged_df))


✅ B8 merged_df: reservation_id duy nhất
✅ B8 row_lineage phủ đủ 10.325 record_id
✅ B8 số dòng kept = số reservation


True

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Nhóm gộp theo `reservation_id`; giá trị đồng thuận được giữ, giá trị thiếu được lấy từ dòng khác nếu có.
> - Nếu các dòng có giá trị khác nhau ở cùng cột, nhóm sắp theo `record_id` tăng dần và chọn giá trị không thiếu đầu tiên; mọi xung đột được lưu trong `b8_conflicts`.
> - Dòng đầu tiên được đánh dấu `kept`, các dòng đã gộp đánh dấu `merged`; dòng đã bị loại ở B1 (nếu có) vẫn nằm trong `row_lineage` với trạng thái `dropped`.
> - Quy tắc này tạo một dòng cho mỗi đặt chỗ mà vẫn giữ đủ truy vết của dữ liệu đầu vào.


---
# GIAI ĐOẠN C · BẢNG SẠCH, BẢNG THÁNG VÀ KIỂM TRA

## C1 · Cột dẫn xuất

| | |
|---|---|
| **Mục tiêu** | Hoàn thiện `clean_df` (1 dòng/reservation_id) với 7 cột dẫn xuất. |
| **Đầu vào → Đầu ra** | `merged_df` → `clean_df` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 8.1 |

**Các bước cần làm**

1. `departure_year`, `departure_month` từ `departure_date` hợp lệ.
2. `total_trip_days = trip_weekend_days + trip_weekdays`; `total_passengers = adult_passengers + child_passengers + infant_passengers`. Nếu thiếu một thành phần → kết quả thiếu.
3. `lead_time_group`: `0-7`, `8-30`, `31-90`, `91-180`, `181+`.
4. `cabin_changed_flag = True` khi cả `booked_cabin_code` và `assigned_cabin_code` có dữ liệu và khác nhau; ngược lại False.
5. `booking_value_est = avg_ticket_fare × total_passengers` **chỉ khi `is_cancelled == 0`** và hai thành phần hợp lệ; còn lại thiếu. Làm tròn 2 chữ số.
6. Khi xuất CSV, hai cột ngày dùng `YYYY-MM-DD`.

`booking_value_est` chỉ là **ước tính giá trị vé cơ sở của reservation**, không phải doanh thu thực tế và không bao gồm thuế/phí/dịch vụ phụ trợ.


In [30]:
# Bắt đầu từ bảng đã gộp ở B8.
# Mỗi dòng trong merged_df là một reservation_id.
clean_df = merged_df.copy()

# 1. Lấy năm và tháng từ ngày khởi hành.
# Nếu departure_date thiếu, năm và tháng cũng sẽ thiếu.
clean_df['departure_year'] = clean_df['departure_date'].dt.year.astype('Int64')
clean_df['departure_month'] = clean_df['departure_date'].dt.month.astype('Int64')

# Đổi các cột dùng trong phép tính về dạng số.
# Việc này tránh lỗi nếu dữ liệu trong kernel vẫn còn là chuỗi.
numeric_columns = [
    'trip_weekend_days', 'trip_weekdays',
    'adult_passengers', 'child_passengers', 'infant_passengers',
    'booking_lead_days', 'is_cancelled', 'avg_ticket_fare',
]

numeric_data = {}

for col in numeric_columns:
    numeric_data[col] = pd.to_numeric(clean_df[col], errors='coerce')
    invalid_values = clean_df[col].notna() & numeric_data[col].isna()

    if invalid_values.any():
        print(
            'Cảnh báo:', invalid_values.sum(),
            'giá trị không phải số trong cột', col,
        )

# 2. Tính tổng số ngày của chuyến đi.
# Nếu một trong hai cột thành phần thiếu, tổng cũng tự thiếu.
clean_df['total_trip_days'] = (
    numeric_data['trip_weekend_days']
    + numeric_data['trip_weekdays']
)

# Tính tổng số hành khách trong mỗi đặt chỗ.
# Không thay giá trị thiếu bằng 0.
clean_df['total_passengers'] = (
    numeric_data['adult_passengers']
    + numeric_data['child_passengers']
    + numeric_data['infant_passengers']
)

# 3. Chia booking_lead_days thành 5 nhóm theo đề bài.
# Chuyển về số để tránh lỗi nếu kernel còn giữ số dưới dạng chuỗi.
lead_days = numeric_data['booking_lead_days']

lead_bins = [-1, 7, 30, 90, 180, float('inf')]
clean_df['lead_time_group'] = pd.cut(
    lead_days,
    bins=lead_bins,
    labels=LEAD_GROUPS,
).astype('string')

# 4. Đánh dấu đổi cabin.
# True chỉ khi biết cả hai mã và chúng khác nhau; các trường hợp khác là False.
both_cabins_known = (
    clean_df['booked_cabin_code'].notna()
    & clean_df['assigned_cabin_code'].notna()
)
clean_df['cabin_changed_flag'] = (
    both_cabins_known
    & clean_df['booked_cabin_code'].ne(clean_df['assigned_cabin_code'])
).fillna(False)

# 5. Chỉ ước tính giá trị vé cho đặt chỗ không bị hủy.
# Cả giá vé và tổng số hành khách phải có dữ liệu.
can_estimate_value = (
    numeric_data['is_cancelled'].eq(0).fillna(False)
    & numeric_data['avg_ticket_fare'].notna()
    & clean_df['total_passengers'].notna()
)
estimated_value = (
    numeric_data['avg_ticket_fare']
    * clean_df['total_passengers']
).round(2)
clean_df['booking_value_est'] = estimated_value.where(can_estimate_value)

# Giữ đúng các cột bắt buộc theo đề bài, theo đúng thứ tự.
clean_df = clean_df[CLEAN_REQUIRED].copy()

print('Kích thước clean_df:', clean_df.shape)
print('Số reservation:', clean_df['reservation_id'].nunique())
display(clean_df.head())

Kích thước clean_df: (10000, 30)
Số reservation: 10000


,reservation_id,airline,is_cancelled,booking_lead_days,departure_date,trip_weekend_days,trip_weekdays,adult_passengers,child_passengers,infant_passengers,meal_option,passenger_country,market_segment,sales_channel,is_returning_customer,booked_cabin_code,assigned_cabin_code,fare_policy,traveler_type,avg_ticket_fare,special_service_requests,booking_status,booking_status_date,departure_year,departure_month,total_trip_days,total_passengers,lead_time_group,cabin_changed_flag,booking_value_est
0,AR002912,Aurora Air,0,19,2024-11-06,4,8,4,0,1,Special,VNM,Offline Agent,Travel Agency,0,F2,F2,Semi-Flexible,Individual,829.51,2,Flown,2024-11-07,2024,11,12,5,8-30,False,4147.55
1,AR008211,Aurora Air,0,0,2024-11-15,2,4,1,0,0,Standard,USA,Online OTA,Website,1,E1,E1,Non-Refundable,Individual,151.68,0,Flown,2024-11-17,2024,11,6,1,0-7,False,151.68
2,AR005564,Aurora Air,0,42,2024-12-12,0,2,2,0,0,Standard,VNM,Corporate,Travel Agency,0,E4,E4,Flexible,Group,146.28,1,Flown,2024-12-12,2024,12,2,2,31-90,False,292.56
3,AR007528,Aurora Air,1,89,2024-08-20,0,4,1,0,0,Special,SWE,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,206.48,0,Cancelled,2024-06-07,2024,8,4,1,31-90,False,<NA>
4,AR007195,Aurora Air,0,18,2025-02-10,0,5,1,1,0,No Meal,GBR,Corporate,GDS/Corporate,0,E2,E2,Semi-Flexible,Corporate Contract,186.89,1,Flown,2025-02-11,2025,2,5,2,8-30,False,373.78


## C2 · Bảng tổng hợp theo tháng

| | |
|---|---|
| **Mục tiêu** | Tạo `monthly_summary`: mỗi dòng là một cặp (hãng, năm khởi hành, tháng khởi hành). |
| **Đầu vào → Đầu ra** | `clean_df` → `monthly_summary` |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 8.2 |

**Các bước cần làm**

1. Chỉ dùng reservation có `departure_date` hợp lệ.
2. `n_reservations` = số reservation; `n_cancelled` = số có `is_cancelled == 1`.
3. `cancel_rate_pct = n_cancelled / số reservation có is_cancelled không thiếu × 100`, làm tròn 2 chữ số.
4. `passengers_flown` = tổng `total_passengers` của reservation **không hủy** (`is_cancelled == 0`).
5. Đúng 7 cột: `airline, departure_year, departure_month, n_reservations, n_cancelled, cancel_rate_pct, passengers_flown`.

**⚠️ Bẫy:** không lấy số dòng `monthly_summary` làm số reservation; không dùng `n_reservations` làm mẫu số tỷ lệ nếu có `is_cancelled` thiếu.


In [31]:
# C2: Tạo bảng tổng hợp theo hãng, năm và tháng khởi hành.
# Mỗi dòng đầu vào của clean_df là một reservation đã gộp ở B8.

# 1. Chỉ giữ reservation có ngày khởi hành hợp lệ.
monthly_data = clean_df.loc[clean_df['departure_date'].notna()].copy()

# 2. Đảm bảo hai cột dùng để tính toán có dạng số.
# errors='coerce' đổi dữ liệu không hợp lệ thành giá trị thiếu.
monthly_data['_status_num'] = pd.to_numeric(
    monthly_data['is_cancelled'],
    errors='coerce',
)
monthly_data['_passengers_num'] = pd.to_numeric(
    monthly_data['total_passengers'],
    errors='coerce',
)

# 3. Chỉ tính hành khách đã bay: reservation không bị hủy.
# Reservation bị hủy hoặc không rõ trạng thái sẽ không được cộng vào.
monthly_data['_passengers_flown'] = monthly_data['_passengers_num'].where(
    monthly_data['_status_num'].eq(0)
)

# 4. Gom dữ liệu theo hãng, năm và tháng khởi hành.
monthly_groups = monthly_data.groupby(
    ['airline', 'departure_year', 'departure_month'],
    as_index=False,
)

# 5. Tính các số liệu cần cho từng nhóm tháng.
monthly_summary = monthly_groups.agg(
    n_reservations=('reservation_id', 'size'),
    n_cancelled=(
        '_status_num',
        lambda status: status.eq(1).sum(),
    ),
    known_status=('_status_num', 'count'),
    passengers_flown=('_passengers_flown', 'sum'),
)

# 6. Tính tỷ lệ hủy trên số reservation có trạng thái đã biết.
# Nếu không có trạng thái nào được biết, tỷ lệ để thiếu thay vì chia cho 0.
monthly_summary['cancel_rate_pct'] = (
    monthly_summary['n_cancelled']
    .div(monthly_summary['known_status'])
    .where(monthly_summary['known_status'].ne(0))
    .mul(100)
    .round(2)
)

# 7. Xóa cột trung gian và sắp xếp đúng 7 cột theo yêu cầu đề bài.
monthly_summary = monthly_summary[SUMMARY_COLUMNS].copy()
monthly_summary = monthly_summary.sort_values(
    ['airline', 'departure_year', 'departure_month']
).reset_index(drop=True)

# Xem kết quả để kiểm tra trước khi chuyển sang bước tiếp theo.
print('Kích thước monthly_summary:', monthly_summary.shape)
print('Số nhóm hãng-năm-tháng:', len(monthly_summary))
display(monthly_summary.head(10))

Kích thước monthly_summary: (52, 7)
Số nhóm hãng-năm-tháng: 52


,airline,departure_year,departure_month,n_reservations,n_cancelled,cancel_rate_pct,passengers_flown
0,Aurora Air,2024,7,243,56,23.05,400
1,Aurora Air,2024,8,218,69,31.65,302
2,Aurora Air,2024,9,159,48,30.19,227
3,Aurora Air,2024,10,182,48,26.37,271
4,Aurora Air,2024,11,190,52,27.37,342
5,Aurora Air,2024,12,217,68,31.34,351
6,Aurora Air,2025,1,337,89,26.41,599
7,Aurora Air,2025,2,299,81,27.09,443
8,Aurora Air,2025,3,266,66,24.81,481
9,Aurora Air,2025,4,277,86,31.05,444


## C3 · Nhật ký và bảng chất lượng trước/sau

| | |
|---|---|
| **Mục tiêu** | Hoàn tất `cleaning_log` từ các lần gọi `log_step` và lập `quality_report`. |
| **Đầu vào → Đầu ra** | `cleaning_log_rows`, `raw_df`, `clean_df` → `cleaning_log`, `quality_report` |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 8.3 |

**Các bước cần làm**

1. `cleaning_log = pd.DataFrame(cleaning_log_rows, columns=LOG_COLUMNS)` — đã có sẵn bên dưới.
2. `quality_report` dạng dài: mỗi cột × `stage` (`before` / `after`), với `denominator_rows` (10.325 hoặc số đặt chỗ), `missing_count`, `invalid_count`, `missing_pct`, có thể thêm `note`.
3. Giải thích được vì sao tỷ lệ thiếu của một số cột **tăng** sau làm sạch.

**Gợi ý hàm:** `pd.DataFrame(list_of_dicts)`, `pd.concat`

**⚠️ Bẫy thường gặp**

- Dùng chung một mẫu số cho trước và sau khi số dòng đã thay đổi.

In [32]:
# 1. Gom các quy tắc đã ghi bằng log_step thành bảng nhật ký.
cleaning_log = pd.DataFrame(
    cleaning_log_rows,
    columns=LOG_COLUMNS,
) if cleaning_log_rows else None

# Chỉ so sánh các cột có ở cả dữ liệu gốc và dữ liệu đã làm sạch.
# Các cột mới tạo ở C1 không có giá trị "before" để so sánh.
quality_columns = [
    col for col in CLEAN_REQUIRED
    if col in raw_df.columns and col in clean_df.columns
]

# Các giá trị hợp lệ theo đề bài, dùng để đếm dữ liệu không hợp lệ.
allowed_labels = {
    'airline': AIRLINES,
    'meal_option': MEALS,
    'market_segment': SEGMENTS,
    'sales_channel': CHANNELS,
    'fare_policy': FARE_POLICIES,
    'traveler_type': TRAVELERS,
    'booking_status': STATUSES,
    'booked_cabin_code': CABIN_CODES,
    'assigned_cabin_code': CABIN_CODES,
}

number_limits = {
    **INT_RULES,
    'is_cancelled': (0, 1),
    'is_returning_customer': (0, 1),
    'avg_ticket_fare': (FARE_MIN, FARE_MAX),
}

def get_invalid_mask(frame, column, has_value):
    """Đánh dấu giá trị có dữ liệu nhưng không đúng quy tắc của cột."""
    values = frame[column]

    # Cột dạng số: kiểm tra chuyển đổi, khoảng giá trị và số nguyên nếu cần.
    if column in number_limits:
        minimum, maximum = number_limits[column]
        numbers = pd.to_numeric(values, errors='coerce')
        valid = numbers.notna() & numbers.between(minimum, maximum)

        if column in INT_RULES or column in ['is_cancelled', 'is_returning_customer']:
            valid = valid & numbers.mod(1).eq(0)

    # Cột ngày: dùng đúng các format và khoảng ngày đã quy định ở B5.
    elif column in ['departure_date', 'booking_status_date']:
        date_formats = (
            DATE_FORMATS if column == 'departure_date'
            else ['%Y-%m-%d']
        )
        parsed_dates = pd.Series(pd.NaT, index=values.index, dtype='datetime64[ns]')

        for date_format in date_formats:
            parsed = pd.to_datetime(values, format=date_format, errors='coerce')
            parsed_dates = parsed_dates.combine_first(parsed)

        if column == 'departure_date':
            valid = parsed_dates.between(DEPARTURE_MIN, DEPARTURE_MAX)
        else:
            valid = parsed_dates.between(STATUS_MIN, STATUS_MAX)

    # Cột phân loại: giá trị phải nằm trong danh sách chuẩn.
    elif column in allowed_labels:
        labels = values.astype('string').str.strip()
        valid = labels.isin(allowed_labels[column])

    # ID và mã quốc gia có quy tắc định dạng riêng.
    elif column == 'reservation_id':
        valid = values.astype('string').str.fullmatch(r'AR\d{6}', na=False)
    elif column == 'passenger_country':
        valid = values.astype('string').str.fullmatch(r'[A-Z]{2,3}', na=False)
    else:
        # Nếu chưa có quy tắc kiểm tra riêng, không tự kết luận là sai.
        valid = pd.Series(True, index=values.index)

    # Chỉ đếm ô có dữ liệu nhưng không hợp lệ; ô thiếu đã được đếm riêng.
    return has_value & ~valid.fillna(False)

# 2. Tính missing_count, invalid_count và missing_pct cho từng cột,
# ở cả dữ liệu trước và sau làm sạch.
quality_rows = []

for stage, frame in [('before', raw_df), ('after', clean_df)]:
    denominator = len(frame)

    for column in quality_columns:
        values = frame[column]
        text_values = values.astype('string').str.strip()
        missing = (
            values.isna()
            | text_values.eq('').fillna(False)
            | text_values.str.upper().isin([token.upper() for token in MISSING_TOKENS])
        )
        invalid = get_invalid_mask(frame, column, ~missing)
        missing_count = int(missing.sum())

        quality_rows.append({
            'column': column,
            'stage': stage,
            'denominator_rows': denominator,
            'missing_count': missing_count,
            'invalid_count': int(invalid.sum()),
            'missing_pct': round(missing_count / denominator * 100, 2)
                if denominator else 0,
        })

# Giữ đúng các cột trong mẫu yêu cầu của đề.
quality_report = pd.DataFrame(quality_rows, columns=QUALITY_COLUMNS)

# 3. So sánh tỷ lệ thiếu trước/sau để nhìn thấy cột nào tăng.
missing_comparison = quality_report.pivot(
    index='column',
    columns='stage',
    values='missing_pct',
).reset_index()
missing_comparison['change_pct_points'] = (
    missing_comparison['after'] - missing_comparison['before']
).round(2)

print('Kích thước cleaning_log:', cleaning_log.shape if cleaning_log is not None else (0, 0))
print('Kích thước quality_report:', quality_report.shape)
display(quality_report.head(10))
print(
    'Lưu ý: tỷ lệ thiếu có thể tăng khi dữ liệu sai định dạng hoặc ngoài miền '
    'được chuyển thành thiếu; mẫu số sau gộp cũng nhỏ hơn số dòng ban đầu.'
)
print('Các cột có tỷ lệ thiếu tăng sau làm sạch:')
display(
    missing_comparison.loc[missing_comparison['change_pct_points'] > 0]
    .sort_values('change_pct_points', ascending=False)
)

Kích thước cleaning_log: (28, 11)
Kích thước quality_report: (46, 6)


,column,stage,denominator_rows,missing_count,invalid_count,missing_pct
0,reservation_id,before,10325,0,187,0.00
1,airline,before,10325,0,139,0.00
2,is_cancelled,before,10325,0,94,0.00
3,booking_lead_days,before,10325,0,55,0.00
4,departure_date,before,10325,25,10,0.24
5,trip_weekend_days,before,10325,12,31,0.12
6,trip_weekdays,before,10325,11,34,0.11
7,adult_passengers,before,10325,12,26,0.12
8,child_passengers,before,10325,40,30,0.39
9,infant_passengers,before,10325,0,11,0.00


Lưu ý: tỷ lệ thiếu có thể tăng khi dữ liệu sai định dạng hoặc ngoài miền được chuyển thành thiếu; mẫu số sau gộp cũng nhỏ hơn số dòng ban đầu.
Các cột có tỷ lệ thiếu tăng sau làm sạch:


stage,column,after,before,change_pct_points
2,assigned_cabin_code,0.40,0.00,0.40
4,booked_cabin_code,0.39,0.00,0.39
6,booking_status,0.23,0.00,0.23
5,booking_lead_days,0.22,0.00,0.22
3,avg_ticket_fare,0.37,0.25,0.12
9,departure_date,0.35,0.24,0.11
14,market_segment,0.09,0.00,0.09
22,trip_weekend_days,0.20,0.12,0.08
21,trip_weekdays,0.19,0.11,0.08
0,adult_passengers,0.19,0.12,0.07


## C4 · Chạy bộ kiểm tra toàn diện

Ô dưới chạy **cùng bộ kiểm tra** với công cụ `kiem_tra_nop_bai.py`. Mục tiêu: **tất cả ✅** trước khi sang giai đoạn D.
Nếu một mục ❌, đọc phần chi tiết để biết số dòng sai, sửa ở bước B/C tương ứng rồi chạy lại.

In [33]:
if validate_outputs is None:
    print('⚠️ Chưa có tools/kiem_tra_nop_bai.py; hãy tải/copy checker vào đúng vị trí để chạy kiểm tra toàn diện.')
else:
    report = validate_outputs(clean_df, monthly_summary, row_lineage, cleaning_log, quality_report,
                              raw_record_ids=raw_df['record_id'])
    all_ok = print_report(report)


✅ [0 Tồn tại] reservations_clean đã được tạo
✅ [0 Tồn tại] monthly_summary đã được tạo
✅ [0 Tồn tại] row_lineage đã được tạo
✅ [0 Tồn tại] cleaning_log đã được tạo
✅ [0 Tồn tại] quality_report đã được tạo
✅ [1 Khóa & nhãn] Đủ các cột bắt buộc
✅ [1 Khóa & nhãn] reservation_id không thiếu, dạng AR + 6 chữ số
✅ [1 Khóa & nhãn] reservation_id không trùng (1 dòng / đặt chỗ)
✅ [1 Khóa & nhãn] airline chỉ gồm Aurora Air, Horizon Wings (không thiếu)
✅ [1 Khóa & nhãn] meal_option thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] market_segment thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] sales_channel thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] fare_policy thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] traveler_type thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] booking_status thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] booked_cabin_code thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] assigned_cabin_code thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] passenge

---
# GIAI ĐOẠN D · TRẢ LỜI 10 CÂU HỎI

**Quy tắc chung cho mọi câu**

- Phân tích từ `clean_df` (đơn vị: **1 đặt chỗ = 1 reservation_id**).
- **Tỷ lệ hủy** = số đặt chỗ `is_cancelled == 1` / số đặt chỗ có `is_cancelled` không thiếu. Ghi rõ mẫu số.
- Mỗi câu: **bảng nguồn** (`save_table`) → **biểu đồ** (`save_fig`, nếu được yêu cầu) → **nhận xét ✍️**.
- Màu theo `AIRLINE_COLORS` / `STATUS_COLORS`; trục số lượng và tỷ lệ của biểu đồ cột bắt đầu từ 0; ghi `n = …` trong ghi chú hình.
- Dùng `finish_chart(ax, title, xlabel, ylabel, note)` để định dạng thống nhất.

## Câu 1 · Chất lượng dữ liệu đã thay đổi thế nào

| | |
|---|---|
| **Cần tính** | So sánh trước/sau: số dòng, số reservation_id, số dòng trùng, tỷ lệ thiếu từng cột. Nêu ≥ 3 quyết định làm sạch ảnh hưởng tới kết quả và **một chỉ tiêu bị sai nếu chưa chuẩn hóa** (ví dụ tỷ lệ hủy tính trên cột `is_cancelled` thô, hoặc số đặt chỗ của Aurora Air khi chưa gộp biến thể tên). |
| **Mẫu số và trình bày** | Mẫu số: 10.325 dòng (trước) và số đặt chỗ (sau). Không bắt buộc biểu đồ. |
| **Đầu ra** | `clean_df` → `q01_before_after` |

**Gợi ý:** Cột gợi ý: `metric, before, after, note`.

In [34]:
q01 = None   # TODO: bảng q01_before_after

save_table(q01, 'q01_before_after')

⏳ Chưa lưu q01_before_after.csv: biến còn None.


In [35]:
check('q01_before_after đã được tạo', lambda: q01 is not None and len(q01) > 0)

❌ q01_before_after đã được tạo


False

> **✍️ Nhận xét Câu 1** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…`*

## Câu 2 · Số đặt chỗ và tỷ lệ hủy theo hãng hàng không

| | |
|---|---|
| **Cần tính** | Số đặt chỗ, tỷ trọng mỗi hãng hàng không, số hủy và **tỷ lệ hủy** theo `airline`; chênh lệch tỷ lệ hủy tính bằng **điểm phần trăm**. |
| **Mẫu số và trình bày** | Biểu đồ 1: cột (tỷ lệ hủy hoặc số đặt chỗ tách Hủy/Không hủy), ghi số trên cột. |
| **Đầu ra** | `clean_df` → `q02_airline_cancel` + `fig01_airline_cancel.png` |

**Gợi ý:** Cột gợi ý: `airline, n_reservations, share_pct, n_cancelled, cancel_rate_pct`.

In [36]:
q02 = None   # TODO: bảng q02_airline_cancel

save_table(q02, 'q02_airline_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q02_airline_cancel.csv')
# save_fig(fig, 'fig01_airline_cancel')
# plt.show()

⏳ Chưa lưu q02_airline_cancel.csv: biến còn None.


In [37]:
check('q02_airline_cancel đã được tạo', lambda: q02 is not None and len(q02) > 0)
check('q02 có đúng 2 hãng hàng không', lambda: len(q02) == 2)
check('Đã lưu fig01_airline_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig01_airline_cancel.png').exists())

❌ q02_airline_cancel đã được tạo
❌ q02 có đúng 2 hãng hàng không — TypeError: object of type 'NoneType' has no len()
❌ Đã lưu fig01_airline_cancel.png


False

> **✍️ Nhận xét Câu 2** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig01_airline_cancel.png`*

## Câu 3 · Mùa vụ: số đặt chỗ theo tháng đến năm 2025

| | |
|---|---|
| **Cần tính** | Chỉ năm **2025** (năm duy nhất đủ 12 tháng trong dữ liệu): số đặt chỗ theo tháng × hãng hàng không; tỷ lệ hủy theo tháng. Nêu tháng cao nhất, thấp nhất của mỗi hãng hàng không. |
| **Mẫu số và trình bày** | Biểu đồ 2: hai đường theo tháng 1–12. Có thể dùng `monthly_summary`. Không so sánh trực tiếp với 2024, 2026 (thiếu tháng). |
| **Đầu ra** | `clean_df` → `q03_monthly_departures_2025` + `fig02_monthly_departures_2025.png` |

**Gợi ý:** Cột gợi ý: `departure_month, Aurora Air, Horizon Wings, total, cancel_rate_pct`.

In [38]:
q03 = None   # TODO: bảng q03_monthly_departures_2025

save_table(q03, 'q03_monthly_departures_2025')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q03_monthly_departures_2025.csv')
# save_fig(fig, 'fig02_monthly_departures_2025')
# plt.show()

⏳ Chưa lưu q03_monthly_departures_2025.csv: biến còn None.


In [39]:
check('q03_monthly_departures_2025 đã được tạo', lambda: q03 is not None and len(q03) > 0)
check('q03 có đủ 12 tháng', lambda: len(q03) == 12)
check('Đã lưu fig02_monthly_departures_2025.png', lambda: (OUTPUT_DIR / 'figures' / 'fig02_monthly_departures_2025.png').exists())

❌ q03_monthly_departures_2025 đã được tạo
❌ q03 có đủ 12 tháng — TypeError: object of type 'NoneType' has no len()
❌ Đã lưu fig02_monthly_departures_2025.png


False

> **✍️ Nhận xét Câu 3** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig02_monthly_departures_2025.png`*

## Câu 4 · Thời gian đặt trước và hủy đặt chỗ

| | |
|---|---|
| **Cần tính** | Thống kê `booking_lead_days` theo hãng (n, mean, median, Q1, Q3, max) và tỷ lệ hủy theo `lead_time_group`. |
| **Mẫu số và trình bày** | Biểu đồ 3: cột tỷ lệ hủy theo 5 nhóm, đúng thứ tự `0-7 → 181+`. Đây là liên hệ quan sát, không phải quan hệ nhân quả. |
| **Đầu ra** | `q04_leadtime_stats.csv`, `q04_leadtime_cancel.csv`, `fig03_leadtime_cancel.png` |


In [40]:
q04 = None   # TODO: bảng q04_leadtime_stats
q04_b = None   # TODO: bảng q04_leadtime_cancel

save_table(q04, 'q04_leadtime_stats')
save_table(q04_b, 'q04_leadtime_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q04_leadtime_stats.csv')
# save_fig(fig, 'fig03_leadtime_cancel')
# plt.show()

⏳ Chưa lưu q04_leadtime_stats.csv: biến còn None.
⏳ Chưa lưu q04_leadtime_cancel.csv: biến còn None.


In [41]:
check('q04_leadtime_stats đã được tạo', lambda: q04 is not None and len(q04) > 0)
check('q04_leadtime_cancel đã được tạo', lambda: q04_b is not None and len(q04_b) > 0)
check('q04_leadtime_cancel có 5 nhóm', lambda: len(q04_b) == 5)
check('Đã lưu fig03_leadtime_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig03_leadtime_cancel.png').exists())

❌ q04_leadtime_stats đã được tạo
❌ q04_leadtime_cancel đã được tạo
❌ q04_leadtime_cancel có 5 nhóm — TypeError: object of type 'NoneType' has no len()
❌ Đã lưu fig03_leadtime_cancel.png


False

> **✍️ Nhận xét Câu 4** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig03_leadtime_cancel.png`*

## Câu 5 · Top 10 mã quốc gia của hành khách/đại diện đặt chỗ

| | |
|---|---|
| **Cần tính** | Đếm reservation theo `passenger_country`; Top 10; tỷ trọng trên tổng reservation; tỷ lệ hủy từng mã. Báo cáo riêng số reservation thiếu quốc gia. Đồng hạng → sắp tiếp theo mã. |
| **Mẫu số và trình bày** | Biểu đồ 4: cột ngang, sắp theo số lượng. |
| **Đầu ra** | `q05_top10_country.csv` + `fig04_top10_country.png` |

**Lưu ý:** `passenger_country` là mã gắn với hành khách/đại diện đặt chỗ trong bộ dữ liệu, không được mặc nhiên diễn giải là quốc tịch pháp lý.


In [42]:
q05 = None   # TODO: bảng q05_top10_country

save_table(q05, 'q05_top10_country')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q05_top10_country.csv')
# save_fig(fig, 'fig04_top10_country')
# plt.show()

⏳ Chưa lưu q05_top10_country.csv: biến còn None.


In [43]:
check('q05_top10_country đã được tạo', lambda: q05 is not None and len(q05) > 0)
check('q05 có 10 quốc gia', lambda: len(q05) == 10)
check('Đã lưu fig04_top10_country.png', lambda: (OUTPUT_DIR / 'figures' / 'fig04_top10_country.png').exists())

❌ q05_top10_country đã được tạo
❌ q05 có 10 quốc gia — TypeError: object of type 'NoneType' has no len()
❌ Đã lưu fig04_top10_country.png


False

> **✍️ Nhận xét Câu 5** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig04_top10_country.png`*

## Câu 6 · Phân khúc thị trường

| | |
|---|---|
| **Cần tính** | Theo `market_segment`: số đặt chỗ, tỷ trọng, tỷ lệ hủy; báo cáo riêng số thiếu phân khúc. |
| **Mẫu số và trình bày** | Biểu đồ 5: cột ngang (số đặt chỗ hoặc tỷ lệ hủy), sắp theo giá trị. Nêu phân khúc nhỏ (ví dụ Package Tour) có mẫu ít → tỷ lệ kém ổn định. |
| **Đầu ra** | `clean_df` → `q06_segment_cancel` + `fig05_segment_cancel.png` |

**Gợi ý:** Thêm cột `n_reservations` cạnh tỷ lệ để người đọc thấy cỡ mẫu.

In [44]:
q06 = None   # TODO: bảng q06_segment_cancel

save_table(q06, 'q06_segment_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q06_segment_cancel.csv')
# save_fig(fig, 'fig05_segment_cancel')
# plt.show()

⏳ Chưa lưu q06_segment_cancel.csv: biến còn None.


In [45]:
check('q06_segment_cancel đã được tạo', lambda: q06 is not None and len(q06) > 0)
check('Đã lưu fig05_segment_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig05_segment_cancel.png').exists())

❌ q06_segment_cancel đã được tạo
❌ Đã lưu fig05_segment_cancel.png


False

> **✍️ Nhận xét Câu 6** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig05_segment_cancel.png`*

## Câu 7 · Chính sách giá vé và trạng thái reservation

| | |
|---|---|
| **Cần tính** | Bảng chéo `fare_policy × booking_status`; tỷ lệ trong từng `fare_policy` cho `Flown`, `Cancelled`, `No-Show`, và `Unknown` nếu trạng thái thiếu. |
| **Mẫu số và trình bày** | Biểu đồ 6: cột chồng 100%. So sánh tỷ lệ hủy theo **điểm phần trăm** và luôn nêu cỡ mẫu của từng nhóm. Không diễn giải thành quan hệ nhân quả. |
| **Đầu ra** | `q07_fare_policy_status.csv` + `fig06_fare_policy_status.png` |

**Gợi ý:** `pd.crosstab(..., normalize='index')`; chỉ dùng `Unknown` ở bước trình bày, không sửa dữ liệu gốc sạch.


In [46]:
q07 = None   # TODO: bảng q07_fare_policy_status

save_table(q07, 'q07_fare_policy_status')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q07_fare_policy_status.csv')
# save_fig(fig, 'fig06_fare_policy_status')
# plt.show()

⏳ Chưa lưu q07_fare_policy_status.csv: biến còn None.


In [47]:
check('q07_fare_policy_status đã được tạo', lambda: q07 is not None and len(q07) > 0)
check('Đã lưu fig06_fare_policy_status.png', lambda: (OUTPUT_DIR / 'figures' / 'fig06_fare_policy_status.png').exists())

❌ q07_fare_policy_status đã được tạo
❌ Đã lưu fig06_fare_policy_status.png


False

> **✍️ Nhận xét Câu 7** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig06_fare_policy_status.png`*

## Câu 8 · Giá vé bình quân trên mỗi hành khách

| | |
|---|---|
| **Cần tính** | Chỉ reservation **không hủy** có `avg_ticket_fare` hợp lệ và > 0: theo hãng — n, mean, median, min, max, Q1, Q3. Báo cáo riêng số giá bằng 0 và số thiếu. |
| **Mẫu số và trình bày** | Biểu đồ 7: histogram giá vé (USD), tách hai hãng. Có thể chọn bin hợp lý theo dữ liệu; không xóa giá cao chỉ vì hiếm nếu vẫn trong miền hợp lệ. |
| **Đầu ra** | `q08_fare_stats_by_airline.csv` + `fig07_fare_hist.png` |

**Gợi ý:** `groupby('airline')['avg_ticket_fare'].describe()`.


In [48]:
q08 = None   # TODO: bảng q08_fare_stats_by_airline

save_table(q08, 'q08_fare_stats_by_airline')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q08_fare_stats_by_airline.csv')
# save_fig(fig, 'fig07_fare_hist')
# plt.show()

⏳ Chưa lưu q08_fare_stats_by_airline.csv: biến còn None.


In [49]:
check('q08_fare_stats_by_airline đã được tạo', lambda: q08 is not None and len(q08) > 0)
check('Đã lưu fig07_fare_hist.png', lambda: (OUTPUT_DIR / 'figures' / 'fig07_fare_hist.png').exists())

❌ q08_fare_stats_by_airline đã được tạo
❌ Đã lưu fig07_fare_hist.png


False

> **✍️ Nhận xét Câu 8** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig07_fare_hist.png`*

## Câu 9 · Độ dài chuyến đi

| | |
|---|---|
| **Cần tính** | Với reservation **không hủy** có `total_trip_days` hợp lệ: phân bố theo số ngày (0, 1, …, 14, `15+`) và hãng; tỷ lệ chuyến 1–3 ngày; báo cáo riêng số reservation có 0 ngày. |
| **Mẫu số và trình bày** | Biểu đồ 8: cột theo số ngày tăng dần, tách hai hãng. |
| **Đầu ra** | `q09_trip_days.csv` + `fig08_trip_days.png` |

**Gợi ý cột:** `trip_days_group, Aurora Air, Horizon Wings, total, pct`.


In [50]:
q09 = None   # TODO: bảng q09_trip_days

save_table(q09, 'q09_trip_days')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt chỗ; loại … vì …. Nguồn: outputs/tables/q09_trip_days.csv')
# save_fig(fig, 'fig08_trip_days')
# plt.show()

⏳ Chưa lưu q09_trip_days.csv: biến còn None.


In [51]:
check('q09_trip_days đã được tạo', lambda: q09 is not None and len(q09) > 0)
check('Đã lưu fig08_trip_days.png', lambda: (OUTPUT_DIR / 'figures' / 'fig08_trip_days.png').exists())

❌ q09_trip_days đã được tạo
❌ Đã lưu fig08_trip_days.png


False

> **✍️ Nhận xét Câu 9** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig08_trip_days.png`*

## Câu 10 · Thay đổi cabin code giữa lúc đặt và khi được gán

| | |
|---|---|
| **Cần tính** | Trong reservation `Flown` có đủ `booked_cabin_code` và `assigned_cabin_code`: tỷ lệ `cabin_changed_flag` theo hãng và theo cabin code đã đặt. |
| **Mẫu số và trình bày** | Mẫu số: reservation `Flown` có đủ hai mã. Biểu đồ tùy chọn `fig09_cabin_change.png`. Không suy ra nguyên nhân thay đổi (upgrade, operational reassignment, v.v.) vì dữ liệu không chứa lý do. |
| **Đầu ra** | `q10_cabin_change_by_airline.csv`, `q10_cabin_change_by_booked_code.csv` |


In [52]:
q10 = None   # TODO: bảng q10_cabin_change_by_airline
q10_b = None   # TODO: bảng q10_cabin_change_by_booked_code

save_table(q10, 'q10_cabin_change_by_airline')
save_table(q10_b, 'q10_cabin_change_by_booked_code')

⏳ Chưa lưu q10_cabin_change_by_airline.csv: biến còn None.
⏳ Chưa lưu q10_cabin_change_by_booked_code.csv: biến còn None.


In [53]:
check('q10_cabin_change_by_airline đã được tạo', lambda: q10 is not None and len(q10) > 0)
check('q10_cabin_change_by_booked_code đã được tạo', lambda: q10_b is not None and len(q10_b) > 0)

❌ q10_cabin_change_by_airline đã được tạo
❌ q10_cabin_change_by_booked_code đã được tạo


False

> **✍️ Nhận xét Câu 10** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt chỗ (đã loại … đặt chỗ vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…`*

---
# 6 INSIGHT CÓ BẰNG CHỨNG

Viết **6 insight** từ **ít nhất 4 chủ đề khác nhau**: hãng hàng không, mùa vụ, lead time, quốc gia, phân khúc, chính sách giá vé, giá vé, độ dài chuyến đi, thay đổi cabin, chất lượng dữ liệu. Mỗi insight **80–120 từ**, theo `templates/insight_template.md`.

| # | Chủ đề | Câu/Bảng/Hình làm bằng chứng | Phát hiện chính |
|---|---|---|---|
| 1 | … | … | … |
| 2 | … | … | … |
| 3 | … | … | … |
| 4 | … | … | … |
| 5 | … | … | … |
| 6 | … | … | … |

**Tự kiểm tra:** có số liệu + mẫu số; có so sánh; nêu ý nghĩa trong phạm vi **hai hãng mô phỏng**; nêu giới hạn; trỏ tới bảng/hình; không dùng ngôn ngữ nhân quả khi dữ liệu chỉ cho thấy liên hệ; không gọi `booking_value_est` là doanh thu thực tế.


---
# GIAI ĐOẠN E · XUẤT DỮ LIỆU VÀ TỰ KIỂM TRA TRƯỚC KHI NỘP

1. Chạy ô xuất bên dưới.
2. **Restart & Run all** — mọi ô phải chạy không lỗi, số thứ tự ô liên tục 1, 2, 3…
3. Xem bảng tiến độ: mục tiêu 100%.
4. (Khuyến nghị) Chạy `python tools/kiem_tra_nop_bai.py <thư mục bài nộp>` trên thư mục đã sắp xếp theo Mục 12 của đề.

In [54]:
outputs = {
    'reservations_clean.csv': clean_df,
    'monthly_summary.csv': monthly_summary,
    'cleaning_log.csv': cleaning_log,
    'row_lineage.csv': row_lineage,
    'quality_report.csv': quality_report,
}
for filename, frame in outputs.items():
    if frame is None:
        print(f'⏳ CHƯA XUẤT {filename}: biến còn None.')
    else:
        frame.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')
        print(f'💾 outputs/{filename}  ({len(frame)} dòng)')

💾 outputs/reservations_clean.csv  (10000 dòng)
💾 outputs/monthly_summary.csv  (52 dòng)
💾 outputs/cleaning_log.csv  (28 dòng)
💾 outputs/row_lineage.csv  (10325 dòng)
💾 outputs/quality_report.csv  (46 dòng)


In [55]:
show_progress()

Tiến độ tệp đầu ra: 7/27 (26%)


,Nhóm,Tệp,Có
0,Dữ liệu,reservations_clean.csv,✅
1,Dữ liệu,monthly_summary.csv,✅
2,Dữ liệu,cleaning_log.csv,✅
3,Dữ liệu,row_lineage.csv,✅
4,Dữ liệu,quality_report.csv,✅
5,Bảng,a_missing_survey.csv,✅
6,Bảng,a_issue_catalog.csv,✅
7,Bảng,q01_before_after.csv,⬜
8,Bảng,q02_airline_cancel.csv,⬜
9,Bảng,q03_monthly_departures_2025.csv,⬜


In [56]:
# (Tùy chọn — Colab) Nén thư mục outputs để tải về máy trước khi phiên Colab kết thúc
import shutil
archive = shutil.make_archive('outputs_backup', 'zip', OUTPUT_DIR)
print('Đã tạo', archive, '— tải về từ bảng Files.')

Đã tạo c:\Users\Ta Van Toan\Desktop\ds\da\Airline_Booking_Sinh_vien\outputs_backup.zip — tải về từ bảng Files.
